In [ ]:
V4: the final pipeline, predictions, and deliverables.

We are going to reconstruct the V3 pipelines for all three models on the locked feature sets, etc.
Evaluate the chosen ensemble once on X_hold (the untouched holdout set for validation)
Produce all the required tables and plots
Then refit all three models on all labeled data and generate the Kaggle submission

In [ ]:
from pathlib import Path
import gc
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    roc_curve,
    precision_recall_curve,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report
)

from xgboost import XGBClassifier

import lightgbm as lgb
from lightgbm import LGBMClassifier

from catboost import CatBoostClassifier

import shap


pd.set_option(
    "display.max_columns",
    200
)

pd.set_option(
    "display.float_format",
    lambda value: f"{value:.6f}"
)

In [ ]:
from pathlib import Path

CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
elif (CURRENT_DIR / "notebooks").exists():
    PROJECT_ROOT = CURRENT_DIR
else:
    raise RuntimeError(
        "Could not identify the project root. "
        "Run this notebook from either the project root or the notebooks folder."
    )

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

print("Project root detected.")
print("Data folder:", DATA_DIR.relative_to(PROJECT_ROOT))
print("Output folder:", OUTPUT_DIR.relative_to(PROJECT_ROOT))

application_train_path = (
    DATA_DIR / "application_train.csv"
)

application_test_path = (
    DATA_DIR / "application_test.csv"
)


print(
    "Training file:",
    application_train_path.resolve()
)

print(
    "Test file:",
    application_test_path.resolve()
)

print(
    "Output directory:",
    OUTPUT_DIR.resolve()
)


assert application_train_path.exists(), (
    f"Could not find {application_train_path}"
)

assert application_test_path.exists(), (
    f"Could not find {application_test_path}"
)

In [ ]:
RANDOM_STATE = 42
HOLDOUT_SIZE = 0.20

np.random.seed(
    RANDOM_STATE
)

In [ ]:
# All models remove the raw housing block.

xgb_final_blocks = [
    "financial_burden",
    "document_contact_summaries"
]

catboost_final_blocks = [
    "financial_burden"
]

lightgbm_final_blocks = [
    "financial_burden",
    "document_contact_summaries"
]


final_history_blocks = [
    "bureau",
    "previous",
    "installments"
]

In [ ]:
final_ensemble_weights = {
    "XGBoost": 1 / 3,
    "LightGBM": 1 / 3,
    "CatBoost": 1 / 3
}


FINAL_ITERATIONS = {
    "XGBoost": 1396,
    "LightGBM": 1740,
    "CatBoost": 2073
}

In [ ]:
XGB_FINAL_SETTINGS = {
    "n_estimators": (
        FINAL_ITERATIONS["XGBoost"]
    ),
    "learning_rate": 0.05,

    "max_depth": 3,
    "min_child_weight": 10,

    "subsample": 0.8,
    "colsample_bytree": 0.8,

    "reg_lambda": 1,
    "reg_alpha": 0,
    "gamma": 0,

    "objective": "binary:logistic",
    "eval_metric": "auc",

    "tree_method": "hist",

    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": 0
}

CATBOOST_FINAL_SETTINGS = {
    "iterations": (
        FINAL_ITERATIONS["CatBoost"]
    ),
    "learning_rate": 0.05,

    "depth": 4,
    "l2_leaf_reg": 3,
    "grow_policy": "SymmetricTree",

    "loss_function": "Logloss",
    "eval_metric": "AUC",

    "random_seed": RANDOM_STATE,
    "thread_count": -1,

    "verbose": False,
    "allow_writing_files": False
}

LIGHTGBM_FINAL_SETTINGS = {
    "objective": "binary",
    "boosting_type": "gbdt",

    "n_estimators": (
        FINAL_ITERATIONS["LightGBM"]
    ),
    "learning_rate": 0.02,

    "num_leaves": 15,
    "min_child_samples": 100,
    "max_depth": -1,

    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,

    "reg_lambda": 1,
    "reg_alpha": 0,
    "min_split_gain": 0,

    "scale_pos_weight": 1.0,

    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": -1
}

In [ ]:
V3_SELECTION_RESULTS = {
    "XGBoost_oof_auc": 0.785042,
    "XGBoost_oof_ap": 0.275291,

    "LightGBM_oof_auc": 0.784731,
    "LightGBM_oof_ap": 0.274246,

    "CatBoost_oof_auc": 0.784321,
    "CatBoost_oof_ap": 0.274353,

    "Final_ensemble_oof_auc": 0.786843,
    "Final_ensemble_oof_ap": 0.278130
}


pd.Series(
    V3_SELECTION_RESULTS,
    name="V3 result"
)

Data reconstruction

The original training data are divided into the same fixed 80% development
set and 20% holdout set used throughout V3.

The holdout labels will not be used for preprocessing, model fitting,
iteration selection, feature selection, or ensemble selection.

In [ ]:
required_data_files = [
    "application_train.csv",
    "application_test.csv",
    "bureau.csv",
    "bureau_balance.csv",
    "previous_application.csv",
    "installments_payments.csv",
]

missing_files = [
    file_name
    for file_name in required_data_files
    if not (DATA_DIR / file_name).exists()
]

if missing_files:
    raise FileNotFoundError(
        "The raw Kaggle Home Credit data are not included in this public repository. "
        f"Missing files: {missing_files}. "
        "To rerun the notebook, download the dataset from Kaggle and place the required files in the data/ folder."
    )

required_aggregate_files = [
    "bureau_agg_v3.pkl",
    "previous_agg_v3.pkl",
    "installments_agg_v3.pkl",
]

missing_aggregate_files = [
    file_name
    for file_name in required_aggregate_files
    if not (DATA_DIR / file_name).exists()
]

if missing_aggregate_files:
    raise FileNotFoundError(
        "Some locally generated aggregate feature files are missing. "
        f"Missing files: {missing_aggregate_files}. "
        "These files are not included in the public repository because they are derived from the raw Kaggle data."
    )

In [ ]:
data = pd.read_csv(
    application_train_path
)

kaggle_test = pd.read_csv(
    application_test_path
)


print(
    "Application train shape:",
    data.shape
)

print(
    "Application test shape:",
    kaggle_test.shape
)

display(
    data.head()
)

In [ ]:
required_train_columns = {
    "SK_ID_CURR",
    "TARGET",
    "DAYS_EMPLOYED"
}

required_test_columns = {
    "SK_ID_CURR",
    "DAYS_EMPLOYED"
}


missing_train_columns = (
    required_train_columns
    - set(data.columns)
)

missing_test_columns = (
    required_test_columns
    - set(kaggle_test.columns)
)


assert not missing_train_columns, (
    "Missing required training columns: "
    f"{missing_train_columns}"
)

assert not missing_test_columns, (
    "Missing required test columns: "
    f"{missing_test_columns}"
)

assert data["SK_ID_CURR"].is_unique
assert kaggle_test["SK_ID_CURR"].is_unique

assert data["TARGET"].notna().all()

assert set(
    data["TARGET"].unique()
).issubset({0, 1})

In [ ]:
X = data.drop(
    columns=[
        "SK_ID_CURR",
        "TARGET"
    ]
).copy()

y = (
    data["TARGET"]
    .astype("int8")
    .copy()
)

data_test = kaggle_test.drop(
    columns=[
        "SK_ID_CURR"
    ]
).copy()


print(
    "Raw training predictors:",
    X.shape
)

print(
    "Raw Kaggle predictors:",
    data_test.shape
)

print(
    f"Overall target rate: "
    f"{y.mean():.4%}"
)

In [ ]:
assert list(X.columns) == list(
    data_test.columns
)

assert len(X) == len(y)

assert "TARGET" not in X.columns
assert "SK_ID_CURR" not in X.columns
assert "SK_ID_CURR" not in data_test.columns

In [ ]:
X_dev, X_hold, y_dev, y_hold = (
    train_test_split(
        X,
        y,
        test_size=HOLDOUT_SIZE,
        random_state=RANDOM_STATE,
        stratify=y
    )
)

In [ ]:
split_summary = pd.DataFrame({
    "dataset": [
        "Full labeled data",
        "Development",
        "Holdout"
    ],

    "rows": [
        len(y),
        len(y_dev),
        len(y_hold)
    ],

    "positive_count": [
        int(y.sum()),
        int(y_dev.sum()),
        int(y_hold.sum())
    ],

    "positive_rate": [
        y.mean(),
        y_dev.mean(),
        y_hold.mean()
    ]
})

split_summary

In [ ]:
id_dev = (
    data.loc[
        X_dev.index,
        "SK_ID_CURR"
    ]
    .astype("int32")
    .copy()
)

id_hold = (
    data.loc[
        X_hold.index,
        "SK_ID_CURR"
    ]
    .astype("int32")
    .copy()
)

id_test = (
    kaggle_test[
        "SK_ID_CURR"
    ]
    .astype("int32")
    .copy()
)

In [ ]:
assert len(id_dev) == len(X_dev)
assert len(id_hold) == len(X_hold)
assert len(id_test) == len(data_test)

assert id_dev.index.equals(
    X_dev.index
)

assert id_hold.index.equals(
    X_hold.index
)

assert id_dev.is_unique
assert id_hold.is_unique
assert id_test.is_unique

assert set(id_dev).isdisjoint(
    set(id_hold)
)

In [ ]:
def correct_days_employed(
    frame
):
    """
    Replace the DAYS_EMPLOYED sentinel value 365243 with NaN
    and add an indicator showing that the sentinel was present.
    """
    corrected = frame.copy()

    corrected[
        "DAYS_EMPLOYED_ANOM"
    ] = (
        corrected["DAYS_EMPLOYED"]
        == 365243
    ).astype("int8")

    corrected[
        "DAYS_EMPLOYED"
    ] = (
        corrected["DAYS_EMPLOYED"]
        .replace(
            365243,
            np.nan
        )
    )

    return corrected

In [ ]:
X_dev = correct_days_employed(
    X_dev
)

X_hold = correct_days_employed(
    X_hold
)

data_test = correct_days_employed(
    data_test
)

In [ ]:
for frame_name, frame in {
    "X_dev": X_dev,
    "X_hold": X_hold,
    "data_test": data_test
}.items():

    print(
        frame_name,
        frame.shape,
        "remaining sentinel values:",
        int(
            (
                frame["DAYS_EMPLOYED"]
                == 365243
            ).sum()
        ),
        "anomaly flags:",
        int(
            frame[
                "DAYS_EMPLOYED_ANOM"
            ].sum()
        )
    )

In [ ]:
assert list(X_dev.columns) == list(
    X_hold.columns
)

assert list(X_dev.columns) == list(
    data_test.columns
)

assert X_dev.shape[1] == 121
assert X_hold.shape[1] == 121
assert data_test.shape[1] == 121

assert X_dev.index.equals(
    y_dev.index
)

assert X_hold.index.equals(
    y_hold.index
)

assert set(X_dev.index).isdisjoint(
    set(X_hold.index)
)

assert not (
    X_dev["DAYS_EMPLOYED"]
    == 365243
).any()

assert not (
    X_hold["DAYS_EMPLOYED"]
    == 365243
).any()

assert not (
    data_test["DAYS_EMPLOYED"]
    == 365243
).any()


print(
    "Data reconstruction complete."
)

print(
    "Development shape:",
    X_dev.shape
)

print(
    "Holdout shape:",
    X_hold.shape
)

print(
    "Kaggle test shape:",
    data_test.shape
)

print(
    f"Development target rate: "
    f"{y_dev.mean():.4%}"
)

print(
    f"Holdout target rate: "
    f"{y_hold.mean():.4%}"
)

Lightweight exploratory data analysis

The exploratory outputs in this section are descriptive deliverables rather
than new model-development experiments.

To preserve the role of the holdout set, the missing-value summary,
target-distribution chart, and correlation analysis use only the development
sample. No feature or model decisions will be changed based on these plots.

In [ ]:
missing_value_summary = pd.DataFrame({
    "feature": X_dev.columns,

    "missing_count": (
        X_dev
        .isna()
        .sum()
        .to_numpy()
    ),

    "missing_fraction": (
        X_dev
        .isna()
        .mean()
        .to_numpy()
    )
})

missing_value_summary[
    "missing_percent"
] = (
    100
    * missing_value_summary[
        "missing_fraction"
    ]
)

missing_value_summary = (
    missing_value_summary
    .sort_values(
        "missing_fraction",
        ascending=False
    )
    .reset_index(drop=True)
)

missing_value_summary.to_csv(
    OUTPUT_DIR
    / "missing_value_summary.csv",
    index=False
)

display(
    missing_value_summary.head(30)
)

In [ ]:
missing_overview = pd.Series({
    "number_of_features": (
        X_dev.shape[1]
    ),

    "features_with_any_missingness": int(
        (
            missing_value_summary[
                "missing_count"
            ] > 0
        ).sum()
    ),

    "features_over_25_percent_missing": int(
        (
            missing_value_summary[
                "missing_fraction"
            ] > 0.25
        ).sum()
    ),

    "features_over_50_percent_missing": int(
        (
            missing_value_summary[
                "missing_fraction"
            ] > 0.50
        ).sum()
    ),

    "maximum_missing_fraction": (
        missing_value_summary[
            "missing_fraction"
        ].max()
    )
})

missing_overview

In [ ]:
top_missing = (
    missing_value_summary
    .head(25)
    .sort_values(
        "missing_percent",
        ascending=True
    )
)

figure, axis = plt.subplots(
    figsize=(10, 9)
)

axis.barh(
    top_missing["feature"],
    top_missing["missing_percent"]
)

axis.set_xlabel(
    "Missing observations (%)"
)

axis.set_ylabel(
    "Feature"
)

axis.set_title(
    "Top 25 application features by missingness\n"
    "Development sample"
)

axis.grid(
    axis="x",
    alpha=0.25
)

figure.tight_layout()

figure.savefig(
    OUTPUT_DIR
    / "missing_value_summary.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(figure)

In [ ]:
target_distribution = (
    y_dev
    .value_counts()
    .sort_index()
    .rename_axis("TARGET")
    .reset_index(name="count")
)

target_distribution[
    "proportion"
] = (
    target_distribution["count"]
    / len(y_dev)
)

target_distribution[
    "label"
] = target_distribution[
    "TARGET"
].map({
    0: "No default",
    1: "Default"
})

target_distribution.to_csv(
    OUTPUT_DIR
    / "target_distribution.csv",
    index=False
)

target_distribution

In [ ]:
development_default_rate = (
    y_dev.mean()
)

print(
    "Development observations:",
    f"{len(y_dev):,}"
)

print(
    "Development defaults:",
    f"{int(y_dev.sum()):,}"
)

print(
    "Development default rate:",
    f"{development_default_rate:.4%}"
)

In [ ]:
figure, axis = plt.subplots(
    figsize=(7, 5)
)

bars = axis.bar(
    target_distribution["label"],
    target_distribution["count"]
)

axis.set_ylabel(
    "Number of applicants"
)

axis.set_xlabel(
    "Observed outcome"
)

axis.set_title(
    "Target distribution in the development sample"
)

axis.grid(
    axis="y",
    alpha=0.25
)

for bar, count, proportion in zip(
    bars,
    target_distribution["count"],
    target_distribution["proportion"]
):
    axis.text(
        bar.get_x()
        + bar.get_width() / 2,

        bar.get_height(),

        (
            f"{count:,}\n"
            f"({proportion:.1%})"
        ),

        ha="center",
        va="bottom"
    )

figure.tight_layout()

figure.savefig(
    OUTPUT_DIR
    / "target_distribution.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(figure)

In [ ]:
correlation_features_requested = [
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",

    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "DAYS_REGISTRATION",
    "DAYS_ID_PUBLISH",

    "CNT_CHILDREN",
    "CNT_FAM_MEMBERS",

    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",

    "REGION_POPULATION_RELATIVE"
]

correlation_features = [
    feature
    for feature in (
        correlation_features_requested
    )
    if feature in X_dev.columns
]

print(
    "Selected correlation features:",
    len(correlation_features)
)

In [ ]:
correlation_frame = (
    X_dev[
        correlation_features
    ]
    .copy()
)

correlation_frame[
    "TARGET"
] = y_dev

correlation_matrix = (
    correlation_frame
    .corr()
)

correlation_matrix.to_csv(
    OUTPUT_DIR
    / "selected_feature_correlations.csv"
)

correlation_matrix.round(3)

In [ ]:
figure, axis = plt.subplots(
    figsize=(12, 10)
)

image = axis.imshow(
    correlation_matrix,
    vmin=-1,
    vmax=1
)

axis.set_xticks(
    np.arange(
        len(correlation_matrix.columns)
    )
)

axis.set_yticks(
    np.arange(
        len(correlation_matrix.index)
    )
)

axis.set_xticklabels(
    correlation_matrix.columns,
    rotation=60,
    ha="right"
)

axis.set_yticklabels(
    correlation_matrix.index
)

for row_index in range(
    correlation_matrix.shape[0]
):
    for column_index in range(
        correlation_matrix.shape[1]
    ):
        value = correlation_matrix.iloc[
            row_index,
            column_index
        ]

        axis.text(
            column_index,
            row_index,
            f"{value:.2f}",
            ha="center",
            va="center",
            fontsize=7
        )

axis.set_title(
    "Correlation heatmap for selected variables\n"
    "Development sample"
)

figure.colorbar(
    image,
    ax=axis,
    fraction=0.046,
    pad=0.04,
    label="Pearson correlation"
)

figure.tight_layout()

figure.savefig(
    OUTPUT_DIR
    / "selected_feature_correlation_heatmap.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(figure)

Final feature reconstruction

The feature pipelines in this section reproduce the decisions locked during
V3. No new feature selection is performed.

All three models:

- remove the raw housing and building-condition block;
- add the five financial-burden variables;
- add bureau, previous-application, and installment-payment aggregates.

XGBoost and LightGBM also include three document/contact summary variables.
CatBoost excludes those three summaries because they did not improve its
development-set ROC AUC.

In [ ]:
categorical_features = (
    X_dev
    .select_dtypes(
        include=[
            "object",
            "string",
            "category"
        ]
    )
    .columns
    .tolist()
)

numerical_features = (
    X_dev
    .select_dtypes(
        include=["number"]
    )
    .columns
    .tolist()
)


contact_flags = [
    "FLAG_MOBIL",
    "FLAG_EMP_PHONE",
    "FLAG_WORK_PHONE",
    "FLAG_CONT_MOBILE",
    "FLAG_PHONE",
    "FLAG_EMAIL"
]

address_flags = [
    "REG_REGION_NOT_LIVE_REGION",
    "REG_REGION_NOT_WORK_REGION",
    "LIVE_REGION_NOT_WORK_REGION",
    "REG_CITY_NOT_LIVE_CITY",
    "REG_CITY_NOT_WORK_CITY",
    "LIVE_CITY_NOT_WORK_CITY"
]

document_flags = [
    f"FLAG_DOCUMENT_{number}"
    for number in range(2, 22)
]


print(
    "Categorical features:",
    len(categorical_features)
)

print(
    "Numerical features:",
    len(numerical_features)
)

In [ ]:
housing_exact_features = {
    "NAME_HOUSING_TYPE",
    "FONDKAPREMONT_MODE",
    "HOUSETYPE_MODE",
    "TOTALAREA_MODE",
    "WALLSMATERIAL_MODE",
    "EMERGENCYSTATE_MODE"
}

housing_prefixes = (
    "APARTMENTS_",
    "BASEMENTAREA_",
    "YEARS_BEGINEXPLUATATION_",
    "YEARS_BUILD_",
    "COMMONAREA_",
    "ELEVATORS_",
    "ENTRANCES_",
    "FLOORSMAX_",
    "FLOORSMIN_",
    "LANDAREA_",
    "LIVINGAPARTMENTS_",
    "LIVINGAREA_",
    "NONLIVINGAPARTMENTS_",
    "NONLIVINGAREA_"
)

housing_ablation_features = [
    feature
    for feature in X_dev.columns
    if (
        feature in housing_exact_features
        or feature.startswith(
            housing_prefixes
        )
    )
]


print(
    "Raw housing features removed:",
    len(housing_ablation_features)
)

assert len(
    housing_ablation_features
) == 48

In [ ]:
def safe_divide(
    numerator,
    denominator
):
    """
    Divide two pandas Series while replacing zero denominators
    and infinite results with NaN.
    """
    denominator = denominator.replace(
        0,
        np.nan
    )

    result = numerator / denominator

    return result.replace(
        [np.inf, -np.inf],
        np.nan
    )

In [ ]:
FINAL_AVAILABLE_BLOCKS = {
    "financial_burden",
    "document_contact_summaries"
}


def build_final_application_features(
    frame,
    added_blocks
):
    """
    Reproduce the locked V3 application-level feature set.

    Baseline:
        Original application variables with the 48-variable
        raw housing block removed.

    Optional retained blocks:
        financial_burden
        document_contact_summaries
    """
    added_blocks = list(
        added_blocks
    )

    unknown_blocks = (
        set(added_blocks)
        - FINAL_AVAILABLE_BLOCKS
    )

    if unknown_blocks:
        raise ValueError(
            "Unknown final application blocks: "
            f"{sorted(unknown_blocks)}"
        )

    housing_columns_present = [
        feature
        for feature in (
            housing_ablation_features
        )
        if feature in frame.columns
    ]

    engineered = frame.drop(
        columns=housing_columns_present
    ).copy()

    if (
        "financial_burden"
        in added_blocks
    ):
        engineered[
            "CREDIT_INCOME_RATIO"
        ] = safe_divide(
            frame["AMT_CREDIT"],
            frame["AMT_INCOME_TOTAL"]
        )

        engineered[
            "ANNUITY_INCOME_RATIO"
        ] = safe_divide(
            frame["AMT_ANNUITY"],
            frame["AMT_INCOME_TOTAL"]
        )

        engineered[
            "ANNUITY_CREDIT_RATIO"
        ] = safe_divide(
            frame["AMT_ANNUITY"],
            frame["AMT_CREDIT"]
        )

        engineered[
            "CREDIT_GOODS_RATIO"
        ] = safe_divide(
            frame["AMT_CREDIT"],
            frame["AMT_GOODS_PRICE"]
        )

        engineered[
            "INCOME_PER_FAMILY_MEMBER"
        ] = safe_divide(
            frame["AMT_INCOME_TOTAL"],
            frame["CNT_FAM_MEMBERS"]
        )

    if (
        "document_contact_summaries"
        in added_blocks
    ):
        selected_document_flags = [
            feature
            for feature in document_flags
            if feature in frame.columns
        ]

        selected_contact_flags = [
            feature
            for feature in contact_flags
            if feature in frame.columns
        ]

        selected_address_flags = [
            feature
            for feature in address_flags
            if feature in frame.columns
        ]

        engineered[
            "DOCUMENT_FLAG_COUNT"
        ] = (
            frame[
                selected_document_flags
            ]
            .fillna(0)
            .sum(axis=1)
            .astype("int16")
        )

        engineered[
            "CONTACT_FLAG_COUNT"
        ] = (
            frame[
                selected_contact_flags
            ]
            .fillna(0)
            .sum(axis=1)
            .astype("int8")
        )

        engineered[
            "ADDRESS_MISMATCH_COUNT"
        ] = (
            frame[
                selected_address_flags
            ]
            .fillna(0)
            .sum(axis=1)
            .astype("int8")
        )

    return engineered

In [ ]:
X_dev_xgb_application = (
    build_final_application_features(
        X_dev,
        added_blocks=xgb_final_blocks
    )
)

X_hold_xgb_application = (
    build_final_application_features(
        X_hold,
        added_blocks=xgb_final_blocks
    )
)

data_test_xgb_application = (
    build_final_application_features(
        data_test,
        added_blocks=xgb_final_blocks
    )
)


X_dev_catboost_application = (
    build_final_application_features(
        X_dev,
        added_blocks=catboost_final_blocks
    )
)

X_hold_catboost_application = (
    build_final_application_features(
        X_hold,
        added_blocks=catboost_final_blocks
    )
)

data_test_catboost_application = (
    build_final_application_features(
        data_test,
        added_blocks=catboost_final_blocks
    )
)


X_dev_lightgbm_application = (
    build_final_application_features(
        X_dev,
        added_blocks=lightgbm_final_blocks
    )
)

X_hold_lightgbm_application = (
    build_final_application_features(
        X_hold,
        added_blocks=lightgbm_final_blocks
    )
)

data_test_lightgbm_application = (
    build_final_application_features(
        data_test,
        added_blocks=lightgbm_final_blocks
    )
)

In [ ]:
application_feature_counts = pd.DataFrame({
    "model": [
        "XGBoost",
        "CatBoost",
        "LightGBM"
    ],

    "development_features": [
        X_dev_xgb_application.shape[1],
        X_dev_catboost_application.shape[1],
        X_dev_lightgbm_application.shape[1]
    ],

    "holdout_features": [
        X_hold_xgb_application.shape[1],
        X_hold_catboost_application.shape[1],
        X_hold_lightgbm_application.shape[1]
    ],

    "test_features": [
        data_test_xgb_application.shape[1],
        data_test_catboost_application.shape[1],
        data_test_lightgbm_application.shape[1]
    ]
})

application_feature_counts

In [ ]:
def safe_history_ratio(
    numerator,
    denominator
):
    denominator = denominator.replace(
        0,
        np.nan
    )

    result = numerator / denominator

    return result.replace(
        [np.inf, -np.inf],
        np.nan
    )


def aggregate_bureau_balance(
    file_path,
    chunksize=2_000_000
):
    """
    Aggregate bureau_balance.csv to one row per SK_ID_BUREAU.
    """
    chunk_results = []

    reader = pd.read_csv(
        file_path,
        usecols=[
            "SK_ID_BUREAU",
            "MONTHS_BALANCE",
            "STATUS"
        ],
        dtype={
            "SK_ID_BUREAU": "int32",
            "MONTHS_BALANCE": "int16",
            "STATUS": "string"
        },
        chunksize=chunksize
    )

    for chunk_number, chunk in enumerate(
        reader,
        start=1
    ):
        print(
            f"Processing bureau-balance "
            f"chunk {chunk_number}"
        )

        status = (
            chunk["STATUS"]
            .fillna("X")
        )

        # Numeric codes 0–5 remain numeric.
        # C, X, and unexpected values become -1.
        status_numeric = (
            pd.to_numeric(
                status,
                errors="coerce"
            )
            .fillna(-1)
        )

        chunk["BB_DELINQUENT"] = (
            status_numeric >= 1
        ).astype("int8")

        chunk["BB_SEVERE"] = (
            status_numeric >= 2
        ).astype("int8")

        chunk["BB_CURRENT"] = (
            status == "0"
        ).astype("int8")

        chunk["BB_CLOSED"] = (
            status == "C"
        ).astype("int8")

        chunk_summary = (
            chunk
            .groupby(
                "SK_ID_BUREAU",
                sort=False
            )
            .agg(
                BB_RECORD_COUNT=(
                    "MONTHS_BALANCE",
                    "size"
                ),
                BB_MONTHS_MIN=(
                    "MONTHS_BALANCE",
                    "min"
                ),
                BB_MONTHS_MAX=(
                    "MONTHS_BALANCE",
                    "max"
                ),
                BB_DELINQUENT_COUNT=(
                    "BB_DELINQUENT",
                    "sum"
                ),
                BB_SEVERE_COUNT=(
                    "BB_SEVERE",
                    "sum"
                ),
                BB_CURRENT_COUNT=(
                    "BB_CURRENT",
                    "sum"
                ),
                BB_CLOSED_COUNT=(
                    "BB_CLOSED",
                    "sum"
                )
            )
        )

        chunk_results.append(
            chunk_summary
        )

        del chunk, chunk_summary
        gc.collect()

    combined = pd.concat(
        chunk_results,
        axis=0
    )

    # An account may cross a chunk boundary.
    combined = (
        combined
        .groupby(level=0)
        .agg(
            BB_RECORD_COUNT=(
                "BB_RECORD_COUNT",
                "sum"
            ),
            BB_MONTHS_MIN=(
                "BB_MONTHS_MIN",
                "min"
            ),
            BB_MONTHS_MAX=(
                "BB_MONTHS_MAX",
                "max"
            ),
            BB_DELINQUENT_COUNT=(
                "BB_DELINQUENT_COUNT",
                "sum"
            ),
            BB_SEVERE_COUNT=(
                "BB_SEVERE_COUNT",
                "sum"
            ),
            BB_CURRENT_COUNT=(
                "BB_CURRENT_COUNT",
                "sum"
            ),
            BB_CLOSED_COUNT=(
                "BB_CLOSED_COUNT",
                "sum"
            )
        )
    )

    combined["BB_DELINQUENT_RATE"] = (
        safe_history_ratio(
            combined["BB_DELINQUENT_COUNT"],
            combined["BB_RECORD_COUNT"]
        )
    )

    combined["BB_SEVERE_RATE"] = (
        safe_history_ratio(
            combined["BB_SEVERE_COUNT"],
            combined["BB_RECORD_COUNT"]
        )
    )

    combined["BB_CURRENT_RATE"] = (
        safe_history_ratio(
            combined["BB_CURRENT_COUNT"],
            combined["BB_RECORD_COUNT"]
        )
    )

    combined["BB_CLOSED_RATE"] = (
        safe_history_ratio(
            combined["BB_CLOSED_COUNT"],
            combined["BB_RECORD_COUNT"]
        )
    )

    return combined.reset_index()

In [ ]:
bureau_balance_agg = (
    aggregate_bureau_balance(
        DATA_DIR / "bureau_balance.csv"
    )
)

print(
    "Bureau-balance aggregate:",
    bureau_balance_agg.shape
)

assert bureau_balance_agg[
    "SK_ID_BUREAU"
].is_unique

In [ ]:
def build_bureau_aggregate(
    bureau_path,
    bureau_balance_aggregate
):
    bureau_columns = [
        "SK_ID_CURR",
        "SK_ID_BUREAU",
        "CREDIT_ACTIVE",
        "DAYS_CREDIT",
        "CREDIT_DAY_OVERDUE",
        "DAYS_CREDIT_ENDDATE",
        "AMT_CREDIT_MAX_OVERDUE",
        "CNT_CREDIT_PROLONG",
        "AMT_CREDIT_SUM",
        "AMT_CREDIT_SUM_DEBT",
        "AMT_CREDIT_SUM_LIMIT",
        "AMT_CREDIT_SUM_OVERDUE",
        "CREDIT_TYPE",
        "DAYS_CREDIT_UPDATE",
        "AMT_ANNUITY"
    ]

    bureau = pd.read_csv(
        bureau_path,
        usecols=bureau_columns,
        low_memory=False
    )

    bureau = bureau.merge(
        bureau_balance_aggregate,
        on="SK_ID_BUREAU",
        how="left",
        validate="one_to_one"
    )

    status = (
        bureau["CREDIT_ACTIVE"]
        .astype("string")
    )

    bureau["BUREAU_IS_ACTIVE"] = (
        status == "Active"
    ).astype("int8")

    bureau["BUREAU_IS_CLOSED"] = (
        status == "Closed"
    ).astype("int8")

    bureau["BUREAU_IS_BAD_DEBT"] = (
        status == "Bad debt"
    ).astype("int8")

    bureau["BUREAU_IS_OVERDUE"] = (
        bureau["CREDIT_DAY_OVERDUE"] > 0
    ).astype("int8")

    bureau[
        "BUREAU_DEBT_CREDIT_RATIO"
    ] = safe_history_ratio(
        bureau["AMT_CREDIT_SUM_DEBT"],
        bureau["AMT_CREDIT_SUM"]
    )

    bureau[
        "BUREAU_CREDIT_DURATION"
    ] = (
        bureau["DAYS_CREDIT_ENDDATE"]
        - bureau["DAYS_CREDIT"]
    )

    aggregate = (
        bureau
        .groupby(
            "SK_ID_CURR",
            sort=False
        )
        .agg(
            BUREAU_RECORD_COUNT=(
                "SK_ID_BUREAU",
                "size"
            ),
            BUREAU_CREDIT_TYPES_NUNIQUE=(
                "CREDIT_TYPE",
                "nunique"
            ),

            BUREAU_ACTIVE_COUNT=(
                "BUREAU_IS_ACTIVE",
                "sum"
            ),
            BUREAU_ACTIVE_RATE=(
                "BUREAU_IS_ACTIVE",
                "mean"
            ),
            BUREAU_CLOSED_COUNT=(
                "BUREAU_IS_CLOSED",
                "sum"
            ),
            BUREAU_CLOSED_RATE=(
                "BUREAU_IS_CLOSED",
                "mean"
            ),
            BUREAU_BAD_DEBT_COUNT=(
                "BUREAU_IS_BAD_DEBT",
                "sum"
            ),
            BUREAU_OVERDUE_COUNT=(
                "BUREAU_IS_OVERDUE",
                "sum"
            ),
            BUREAU_OVERDUE_RATE=(
                "BUREAU_IS_OVERDUE",
                "mean"
            ),

            BUREAU_DAYS_CREDIT_MEAN=(
                "DAYS_CREDIT",
                "mean"
            ),
            BUREAU_DAYS_CREDIT_MAX=(
                "DAYS_CREDIT",
                "max"
            ),
            BUREAU_DAYS_UPDATE_MEAN=(
                "DAYS_CREDIT_UPDATE",
                "mean"
            ),
            BUREAU_CREDIT_DURATION_MEAN=(
                "BUREAU_CREDIT_DURATION",
                "mean"
            ),

            BUREAU_CREDIT_SUM_SUM=(
                "AMT_CREDIT_SUM",
                "sum"
            ),
            BUREAU_CREDIT_SUM_MEAN=(
                "AMT_CREDIT_SUM",
                "mean"
            ),
            BUREAU_CREDIT_SUM_MAX=(
                "AMT_CREDIT_SUM",
                "max"
            ),

            BUREAU_DEBT_SUM=(
                "AMT_CREDIT_SUM_DEBT",
                "sum"
            ),
            BUREAU_DEBT_MEAN=(
                "AMT_CREDIT_SUM_DEBT",
                "mean"
            ),
            BUREAU_DEBT_MAX=(
                "AMT_CREDIT_SUM_DEBT",
                "max"
            ),

            BUREAU_OVERDUE_AMOUNT_SUM=(
                "AMT_CREDIT_SUM_OVERDUE",
                "sum"
            ),
            BUREAU_OVERDUE_AMOUNT_MAX=(
                "AMT_CREDIT_SUM_OVERDUE",
                "max"
            ),

            BUREAU_DEBT_CREDIT_RATIO_MEAN=(
                "BUREAU_DEBT_CREDIT_RATIO",
                "mean"
            ),
            BUREAU_DEBT_CREDIT_RATIO_MAX=(
                "BUREAU_DEBT_CREDIT_RATIO",
                "max"
            ),

            BUREAU_PROLONG_COUNT_SUM=(
                "CNT_CREDIT_PROLONG",
                "sum"
            ),
            BUREAU_ANNUITY_MEAN=(
                "AMT_ANNUITY",
                "mean"
            ),

            BB_RECORD_COUNT_SUM=(
                "BB_RECORD_COUNT",
                "sum"
            ),
            BB_DELINQUENT_COUNT_SUM=(
                "BB_DELINQUENT_COUNT",
                "sum"
            ),
            BB_DELINQUENT_RATE_MEAN=(
                "BB_DELINQUENT_RATE",
                "mean"
            ),
            BB_DELINQUENT_RATE_MAX=(
                "BB_DELINQUENT_RATE",
                "max"
            ),
            BB_SEVERE_COUNT_SUM=(
                "BB_SEVERE_COUNT",
                "sum"
            ),
            BB_SEVERE_RATE_MEAN=(
                "BB_SEVERE_RATE",
                "mean"
            ),
            BB_CURRENT_RATE_MEAN=(
                "BB_CURRENT_RATE",
                "mean"
            ),
            BB_CLOSED_RATE_MEAN=(
                "BB_CLOSED_RATE",
                "mean"
            )
        )
        .reset_index()
    )

    aggregate[
        "BUREAU_TOTAL_DEBT_CREDIT_RATIO"
    ] = safe_history_ratio(
        aggregate["BUREAU_DEBT_SUM"],
        aggregate["BUREAU_CREDIT_SUM_SUM"]
    )

    del bureau
    gc.collect()

    return aggregate

In [ ]:
bureau_agg = build_bureau_aggregate(
    bureau_path=(
        DATA_DIR / "bureau.csv"
    ),
    bureau_balance_aggregate=(
        bureau_balance_agg
    )
)

print(
    "Final bureau aggregate:",
    bureau_agg.shape
)

assert bureau_agg[
    "SK_ID_CURR"
].is_unique

assert bureau_agg.shape[1] == 35

In [ ]:
bureau_agg.to_pickle(
    DATA_DIR / "bureau_agg_v3.pkl"
)

print(
    "Saved:",
    (
        DATA_DIR
        / "bureau_agg_v3.pkl"
    ).resolve()
)

In [ ]:
bureau_aggregate_path = (
    DATA_DIR / "bureau_agg_v3.pkl"
)

previous_aggregate_path = (
    DATA_DIR / "previous_agg_v3.pkl"
)

installments_aggregate_path = (
    DATA_DIR / "installments_agg_v3.pkl"
)


for aggregate_path in [
    bureau_aggregate_path,
    previous_aggregate_path,
    installments_aggregate_path
]:
    assert aggregate_path.exists(), (
        f"Missing aggregate file: "
        f"{aggregate_path}"
    )


bureau_agg = pd.read_pickle(
    bureau_aggregate_path
)

previous_agg = pd.read_pickle(
    previous_aggregate_path
)

installments_agg = pd.read_pickle(
    installments_aggregate_path
)

In [ ]:
aggregate_summary = pd.DataFrame({
    "block": [
        "Bureau",
        "Previous applications",
        "Installments"
    ],

    "rows": [
        bureau_agg.shape[0],
        previous_agg.shape[0],
        installments_agg.shape[0]
    ],

    "columns_including_id": [
        bureau_agg.shape[1],
        previous_agg.shape[1],
        installments_agg.shape[1]
    ],

    "unique_applicant_ids": [
        bureau_agg[
            "SK_ID_CURR"
        ].is_unique,

        previous_agg[
            "SK_ID_CURR"
        ].is_unique,

        installments_agg[
            "SK_ID_CURR"
        ].is_unique
    ]
})

aggregate_summary

In [ ]:
assert bureau_agg[
    "SK_ID_CURR"
].is_unique

assert previous_agg[
    "SK_ID_CURR"
].is_unique

assert installments_agg[
    "SK_ID_CURR"
].is_unique

In [ ]:
def merge_history_block(
    base_frame,
    id_series,
    aggregate_frame,
    block_prefix,
    record_count_column
):
    """
    Merge a one-row-per-applicant historical aggregate onto an
    application feature frame while preserving its row order
    and index.

    SK_ID_CURR is used only as a merge key and is not returned
    as a model predictor.
    """
    if not aggregate_frame[
        "SK_ID_CURR"
    ].is_unique:
        raise ValueError(
            f"{block_prefix} aggregate has "
            "duplicate applicant IDs."
        )

    aligned_ids = id_series.reindex(
        base_frame.index
    )

    if aligned_ids.isna().any():
        raise ValueError(
            f"{block_prefix}: applicant IDs "
            "do not align with the feature frame."
        )

    history_features = (
        aggregate_frame
        .set_index("SK_ID_CURR")
        .reindex(
            aligned_ids.to_numpy()
        )
    )

    history_features.index = (
        base_frame.index
    )

    history_features[
        f"{block_prefix}_HAS_HISTORY"
    ] = (
        history_features[
            record_count_column
        ]
        .notna()
        .astype("int8")
    )

    # When no historical records exist, counts and totals are
    # genuinely zero. Means, maxima, rates, and ratios remain
    # missing because they are undefined.
    zero_fill_columns = [
        column
        for column in history_features.columns
        if (
            column.endswith("_COUNT")
            or column.endswith("_SUM")
            or column.endswith("_NUNIQUE")
        )
    ]

    history_features[
        zero_fill_columns
    ] = (
        history_features[
            zero_fill_columns
        ]
        .fillna(0)
    )

    overlapping_columns = (
        set(base_frame.columns)
        & set(history_features.columns)
    )

    if overlapping_columns:
        raise ValueError(
            "Duplicate feature names found: "
            f"{sorted(overlapping_columns)}"
        )

    result = pd.concat(
        [
            base_frame,
            history_features
        ],
        axis=1
    )

    if not result.index.equals(
        base_frame.index
    ):
        raise ValueError(
            "Row index changed during "
            "historical merge."
        )

    return result

In [ ]:
HISTORY_BLOCKS = {
    "bureau": {
        "aggregate": bureau_agg,
        "prefix": "BUREAU",
        "record_count": (
            "BUREAU_RECORD_COUNT"
        )
    },

    "previous": {
        "aggregate": previous_agg,
        "prefix": "PREV",
        "record_count": (
            "PREV_APPLICATION_COUNT"
        )
    },

    "installments": {
        "aggregate": installments_agg,
        "prefix": "INST",
        "record_count": (
            "INST_RECORD_COUNT"
        )
    }
}


def attach_selected_history_blocks(
    base_frame,
    id_series,
    selected_blocks
):
    result = base_frame.copy()

    for block_name in selected_blocks:
        if block_name not in HISTORY_BLOCKS:
            raise ValueError(
                "Unknown historical block: "
                f"{block_name}"
            )

        block_information = (
            HISTORY_BLOCKS[
                block_name
            ]
        )

        result = merge_history_block(
            base_frame=result,
            id_series=id_series,
            aggregate_frame=(
                block_information[
                    "aggregate"
                ]
            ),
            block_prefix=(
                block_information[
                    "prefix"
                ]
            ),
            record_count_column=(
                block_information[
                    "record_count"
                ]
            )
        )

    return result

In [ ]:
X_dev_xgb_final = (
    attach_selected_history_blocks(
        base_frame=X_dev_xgb_application,
        id_series=id_dev,
        selected_blocks=final_history_blocks
    )
)

X_hold_xgb_final = (
    attach_selected_history_blocks(
        base_frame=X_hold_xgb_application,
        id_series=id_hold,
        selected_blocks=final_history_blocks
    )
)

data_test_xgb_final = (
    attach_selected_history_blocks(
        base_frame=data_test_xgb_application,
        id_series=id_test,
        selected_blocks=final_history_blocks
    )
)


X_dev_catboost_final = (
    attach_selected_history_blocks(
        base_frame=X_dev_catboost_application,
        id_series=id_dev,
        selected_blocks=final_history_blocks
    )
)

X_hold_catboost_final = (
    attach_selected_history_blocks(
        base_frame=X_hold_catboost_application,
        id_series=id_hold,
        selected_blocks=final_history_blocks
    )
)

data_test_catboost_final = (
    attach_selected_history_blocks(
        base_frame=data_test_catboost_application,
        id_series=id_test,
        selected_blocks=final_history_blocks
    )
)


X_dev_lightgbm_final = (
    attach_selected_history_blocks(
        base_frame=X_dev_lightgbm_application,
        id_series=id_dev,
        selected_blocks=final_history_blocks
    )
)

X_hold_lightgbm_final = (
    attach_selected_history_blocks(
        base_frame=X_hold_lightgbm_application,
        id_series=id_hold,
        selected_blocks=final_history_blocks
    )
)

data_test_lightgbm_final = (
    attach_selected_history_blocks(
        base_frame=data_test_lightgbm_application,
        id_series=id_test,
        selected_blocks=final_history_blocks
    )
)

In [ ]:
final_feature_counts = pd.DataFrame({
    "model": [
        "XGBoost",
        "CatBoost",
        "LightGBM"
    ],

    "development_rows": [
        X_dev_xgb_final.shape[0],
        X_dev_catboost_final.shape[0],
        X_dev_lightgbm_final.shape[0]
    ],

    "number_of_features": [
        X_dev_xgb_final.shape[1],
        X_dev_catboost_final.shape[1],
        X_dev_lightgbm_final.shape[1]
    ]
})

final_feature_counts

In [ ]:
assert X_dev_xgb_final.shape == (
    len(y_dev),
    160
)

assert X_hold_xgb_final.shape == (
    len(y_hold),
    160
)

assert data_test_xgb_final.shape == (
    len(id_test),
    160
)


assert X_dev_catboost_final.shape == (
    len(y_dev),
    157
)

assert X_hold_catboost_final.shape == (
    len(y_hold),
    157
)

assert data_test_catboost_final.shape == (
    len(id_test),
    157
)


assert X_dev_lightgbm_final.shape == (
    len(y_dev),
    160
)

assert X_hold_lightgbm_final.shape == (
    len(y_hold),
    160
)

assert data_test_lightgbm_final.shape == (
    len(id_test),
    160
)

assert list(
    X_dev_xgb_final.columns
) == list(
    X_hold_xgb_final.columns
) == list(
    data_test_xgb_final.columns
)

assert list(
    X_dev_catboost_final.columns
) == list(
    X_hold_catboost_final.columns
) == list(
    data_test_catboost_final.columns
)

assert list(
    X_dev_lightgbm_final.columns
) == list(
    X_hold_lightgbm_final.columns
) == list(
    data_test_lightgbm_final.columns
)

assert X_dev_xgb_final.index.equals(
    y_dev.index
)

assert X_hold_xgb_final.index.equals(
    y_hold.index
)

print(
    "Final feature reconstruction complete."
)

Fit the locked final models

Each model is trained once on the complete development sample.

The number of boosting iterations was fixed using the mean early-stopping
iteration from the final V3 five-fold cross-validation. The holdout set is
not used for early stopping or any other model-selection decision.
Cell 5.1: XGBoost preprocessor

In [ ]:
def build_xgb_preprocessor(
    selected_features,
    categorical_features
):
    """
    Numerical variables pass through unchanged because XGBoost
    handles numerical NaN values natively.

    Categorical variables receive a missing category followed by
    one-hot encoding.
    """
    selected_features = list(
        selected_features
    )

    categorical_feature_set = set(
        categorical_features
    )

    selected_categorical = [
        feature
        for feature in selected_features
        if feature in (
            categorical_feature_set
        )
    ]

    selected_numerical = [
        feature
        for feature in selected_features
        if feature not in (
            categorical_feature_set
        )
    ]

    transformers = []

    if selected_numerical:
        transformers.append(
            (
                "numeric",
                "passthrough",
                selected_numerical
            )
        )

    if selected_categorical:
        categorical_pipeline = Pipeline([
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="Missing"
                )
            ),

            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True
                )
            )
        ])

        transformers.append(
            (
                "categorical",
                categorical_pipeline,
                selected_categorical
            )
        )

    return ColumnTransformer(
        transformers=transformers,
        sparse_threshold=1.0
    )

In [ ]:
def prepare_catboost_pair(
    training_frame,
    prediction_frame,
    categorical_features
):
    """
    Prepare one training frame and one prediction frame for
    native CatBoost categorical handling.
    """
    training_prepared = (
        training_frame.copy()
    )

    prediction_prepared = (
        prediction_frame.copy()
    )

    selected_categorical = [
        feature
        for feature in categorical_features
        if feature in (
            training_prepared.columns
        )
    ]

    for feature in selected_categorical:
        training_prepared[
            feature
        ] = (
            training_prepared[feature]
            .astype("object")
            .where(
                training_prepared[
                    feature
                ].notna(),
                "Missing"
            )
            .astype(str)
        )

        prediction_prepared[
            feature
        ] = (
            prediction_prepared[feature]
            .astype("object")
            .where(
                prediction_prepared[
                    feature
                ].notna(),
                "Missing"
            )
            .astype(str)
        )

    return (
        training_prepared,
        prediction_prepared,
        selected_categorical
    )

In [ ]:
def prepare_lightgbm_pair(
    training_frame,
    prediction_frame,
    categorical_features
):
    """
    Apply development-set category levels to another frame.

    Categories not observed during training are mapped to the
    explicit Missing category.
    """
    training_prepared = (
        training_frame.copy()
    )

    prediction_prepared = (
        prediction_frame.copy()
    )

    selected_categorical = [
        feature
        for feature in categorical_features
        if feature in (
            training_prepared.columns
        )
    ]

    for feature in selected_categorical:
        training_values = (
            training_prepared[feature]
            .astype("string")
            .fillna("Missing")
        )

        prediction_values = (
            prediction_prepared[feature]
            .astype("string")
            .fillna("Missing")
        )

        training_categories = pd.Index(
            training_values.unique()
        )

        if (
            "Missing"
            not in training_categories
        ):
            training_categories = (
                training_categories.append(
                    pd.Index(["Missing"])
                )
            )

        prediction_values = (
            prediction_values.where(
                prediction_values.isin(
                    training_categories
                ),
                "Missing"
            )
        )

        training_prepared[
            feature
        ] = pd.Categorical(
            training_values,
            categories=training_categories
        )

        prediction_prepared[
            feature
        ] = pd.Categorical(
            prediction_values,
            categories=training_categories
        )

    return (
        training_prepared,
        prediction_prepared,
        selected_categorical
    )

In [ ]:
def prepare_lightgbm_pair(
    training_frame,
    prediction_frame,
    categorical_features
):
    """
    Apply development-set category levels to another frame.

    Categories not observed during training are mapped to the
    explicit Missing category.
    """
    training_prepared = (
        training_frame.copy()
    )

    prediction_prepared = (
        prediction_frame.copy()
    )

    selected_categorical = [
        feature
        for feature in categorical_features
        if feature in (
            training_prepared.columns
        )
    ]

    for feature in selected_categorical:
        training_values = (
            training_prepared[feature]
            .astype("string")
            .fillna("Missing")
        )

        prediction_values = (
            prediction_prepared[feature]
            .astype("string")
            .fillna("Missing")
        )

        training_categories = pd.Index(
            training_values.unique()
        )

        if (
            "Missing"
            not in training_categories
        ):
            training_categories = (
                training_categories.append(
                    pd.Index(["Missing"])
                )
            )

        prediction_values = (
            prediction_values.where(
                prediction_values.isin(
                    training_categories
                ),
                "Missing"
            )
        )

        training_prepared[
            feature
        ] = pd.Categorical(
            training_values,
            categories=training_categories
        )

        prediction_prepared[
            feature
        ] = pd.Categorical(
            prediction_values,
            categories=training_categories
        )

    return (
        training_prepared,
        prediction_prepared,
        selected_categorical
    )

In [ ]:
xgb_preprocessor = (
    build_xgb_preprocessor(
        selected_features=(
            X_dev_xgb_final.columns
        ),
        categorical_features=(
            categorical_features
        )
    )
)

xgb_training_matrix = (
    xgb_preprocessor.fit_transform(
        X_dev_xgb_final
    )
)

xgb_holdout_matrix = (
    xgb_preprocessor.transform(
        X_hold_xgb_final
    )
)


print(
    "XGBoost processed development shape:",
    xgb_training_matrix.shape
)

print(
    "XGBoost processed holdout shape:",
    xgb_holdout_matrix.shape
)

In [ ]:
xgb_final_model = XGBClassifier(
    **XGB_FINAL_SETTINGS
)

xgb_start_time = (
    time.perf_counter()
)

xgb_final_model.fit(
    xgb_training_matrix,
    y_dev,
    verbose=False
)

xgb_training_seconds = (
    time.perf_counter()
    - xgb_start_time
)

xgb_holdout_probability = (
    xgb_final_model.predict_proba(
        xgb_holdout_matrix
    )[:, 1]
)

print(
    "XGBoost training time:",
    f"{xgb_training_seconds:.1f} seconds"
)

In [ ]:
xgb_processed_feature_names = (
    xgb_preprocessor
    .get_feature_names_out()
)

assert len(
    xgb_processed_feature_names
) == xgb_training_matrix.shape[1]

In [ ]:
(
    X_dev_catboost_prepared,
    X_hold_catboost_prepared,
    catboost_categorical_features
) = prepare_catboost_pair(
    training_frame=(
        X_dev_catboost_final
    ),
    prediction_frame=(
        X_hold_catboost_final
    ),
    categorical_features=(
        categorical_features
    )
)

In [ ]:
catboost_final_model = (
    CatBoostClassifier(
        **CATBOOST_FINAL_SETTINGS
    )
)

catboost_start_time = (
    time.perf_counter()
)

catboost_final_model.fit(
    X_dev_catboost_prepared,
    y_dev,
    cat_features=(
        catboost_categorical_features
    ),
    verbose=False
)

catboost_training_seconds = (
    time.perf_counter()
    - catboost_start_time
)

catboost_holdout_probability = (
    catboost_final_model.predict_proba(
        X_hold_catboost_prepared
    )[:, 1]
)

print(
    "CatBoost training time:",
    f"{catboost_training_seconds:.1f} seconds"
)

In [ ]:
(
    X_dev_lightgbm_prepared,
    X_hold_lightgbm_prepared,
    lightgbm_categorical_features
) = prepare_lightgbm_pair(
    training_frame=(
        X_dev_lightgbm_final
    ),
    prediction_frame=(
        X_hold_lightgbm_final
    ),
    categorical_features=(
        categorical_features
    )
)

In [ ]:
lightgbm_final_model = (
    LGBMClassifier(
        **LIGHTGBM_FINAL_SETTINGS
    )
)

lightgbm_start_time = (
    time.perf_counter()
)

lightgbm_final_model.fit(
    X_dev_lightgbm_prepared,
    y_dev,
    categorical_feature=(
        lightgbm_categorical_features
    )
)

lightgbm_training_seconds = (
    time.perf_counter()
    - lightgbm_start_time
)

lightgbm_holdout_probability = (
    lightgbm_final_model.predict_proba(
        X_hold_lightgbm_prepared
    )[:, 1]
)

print(
    "LightGBM training time:",
    f"{lightgbm_training_seconds:.1f} seconds"
)

In [ ]:
holdout_probability_dictionary = {
    "XGBoost": (
        xgb_holdout_probability
    ),

    "LightGBM": (
        lightgbm_holdout_probability
    ),

    "CatBoost": (
        catboost_holdout_probability
    )
}


for model_name, probability in (
    holdout_probability_dictionary.items()
):
    assert len(probability) == len(
        y_hold
    )

    assert np.isfinite(
        probability
    ).all()

    assert (
        (
            probability >= 0
        )
        &
        (
            probability <= 1
        )
    ).all()

    print(
        model_name,
        "minimum probability:",
        f"{probability.min():.6f}",
        "maximum probability:",
        f"{probability.max():.6f}"
    )

In [ ]:
final_training_runtime = pd.DataFrame({
    "model": [
        "XGBoost",
        "LightGBM",
        "CatBoost"
    ],

    "fixed_iterations": [
        FINAL_ITERATIONS["XGBoost"],
        FINAL_ITERATIONS["LightGBM"],
        FINAL_ITERATIONS["CatBoost"]
    ],

    "training_seconds": [
        xgb_training_seconds,
        lightgbm_training_seconds,
        catboost_training_seconds
    ]
})

final_training_runtime.to_csv(
    OUTPUT_DIR
    / "final_training_runtime.csv",
    index=False
)

final_training_runtime

Final untouched holdout evaluation

The individual models, feature sets, iteration counts, and equal ensemble
weights were selected before inspecting holdout performance.

This section evaluates the locked system once on the untouched holdout set.
The resulting metrics are final out-of-sample estimates and will not be used
to revise the pipeline.

In [ ]:
final_holdout_probability = (
    final_ensemble_weights[
        "XGBoost"
    ]
    * xgb_holdout_probability

    + final_ensemble_weights[
        "LightGBM"
    ]
    * lightgbm_holdout_probability

    + final_ensemble_weights[
        "CatBoost"
    ]
    * catboost_holdout_probability
)


assert len(
    final_holdout_probability
) == len(y_hold)

assert np.isfinite(
    final_holdout_probability
).all()

assert (
    (
        final_holdout_probability >= 0
    )
    &
    (
        final_holdout_probability <= 1
    )
).all()

In [ ]:
holdout_candidates = {
    "XGBoost": (
        xgb_holdout_probability
    ),

    "LightGBM": (
        lightgbm_holdout_probability
    ),

    "CatBoost": (
        catboost_holdout_probability
    ),

    "Equal-weight ensemble": (
        final_holdout_probability
    )
}


holdout_performance = pd.DataFrame([
    {
        "candidate": candidate_name,

        "holdout_roc_auc": (
            roc_auc_score(
                y_hold,
                probability
            )
        ),

        "holdout_average_precision": (
            average_precision_score(
                y_hold,
                probability
            )
        )
    }

    for candidate_name, probability
    in holdout_candidates.items()
])


holdout_performance = (
    holdout_performance
    .sort_values(
        "holdout_roc_auc",
        ascending=False
    )
    .reset_index(drop=True)
)

holdout_performance

In [ ]:
v3_oof_metrics = pd.DataFrame({
    "candidate": [
        "XGBoost",
        "LightGBM",
        "CatBoost",
        "Equal-weight ensemble"
    ],

    "development_oof_roc_auc": [
        V3_SELECTION_RESULTS[
            "XGBoost_oof_auc"
        ],

        V3_SELECTION_RESULTS[
            "LightGBM_oof_auc"
        ],

        V3_SELECTION_RESULTS[
            "CatBoost_oof_auc"
        ],

        V3_SELECTION_RESULTS[
            "Final_ensemble_oof_auc"
        ]
    ],

    "development_oof_average_precision": [
        V3_SELECTION_RESULTS[
            "XGBoost_oof_ap"
        ],

        V3_SELECTION_RESULTS[
            "LightGBM_oof_ap"
        ],

        V3_SELECTION_RESULTS[
            "CatBoost_oof_ap"
        ],

        V3_SELECTION_RESULTS[
            "Final_ensemble_oof_ap"
        ]
    ]
})


final_evaluation_table = (
    v3_oof_metrics
    .merge(
        holdout_performance,
        on="candidate",
        how="left",
        validate="one_to_one"
    )
)


final_evaluation_table[
    "holdout_minus_oof_auc"
] = (
    final_evaluation_table[
        "holdout_roc_auc"
    ]
    - final_evaluation_table[
        "development_oof_roc_auc"
    ]
)


final_evaluation_table[
    "holdout_minus_oof_ap"
] = (
    final_evaluation_table[
        "holdout_average_precision"
    ]
    - final_evaluation_table[
        "development_oof_average_precision"
    ]
)


final_evaluation_table = (
    final_evaluation_table
    .sort_values(
        "holdout_roc_auc",
        ascending=False
    )
    .reset_index(drop=True)
)

final_evaluation_table

In [ ]:
final_ensemble_row = (
    final_evaluation_table
    .loc[
        final_evaluation_table[
            "candidate"
        ]
        == "Equal-weight ensemble"
    ]
    .iloc[0]
)


final_holdout_auc = (
    final_ensemble_row[
        "holdout_roc_auc"
    ]
)

final_holdout_ap = (
    final_ensemble_row[
        "holdout_average_precision"
    ]
)


print(
    "=" * 55
)

print(
    "FINAL LOCKED ENSEMBLE HOLDOUT RESULTS"
)

print(
    "=" * 55
)

print(
    "Holdout observations:",
    f"{len(y_hold):,}"
)

print(
    "Holdout defaults:",
    f"{int(y_hold.sum()):,}"
)

print(
    "Holdout default rate:",
    f"{y_hold.mean():.4%}"
)

print()

print(
    "Final holdout ROC AUC:",
    f"{final_holdout_auc:.6f}"
)

print(
    "Final holdout average precision:",
    f"{final_holdout_ap:.6f}"
)

print()

print(
    "Development OOF ROC AUC:",
    (
        f"{V3_SELECTION_RESULTS[
            'Final_ensemble_oof_auc'
        ]:.6f}"
    )
)

print(
    "Development OOF average precision:",
    (
        f"{V3_SELECTION_RESULTS[
            'Final_ensemble_oof_ap'
        ]:.6f}"
    )
)

print(
    "=" * 55
)

In [ ]:
holdout_performance.to_csv(
    OUTPUT_DIR
    / "holdout_model_performance.csv",
    index=False
)

final_evaluation_table.to_csv(
    OUTPUT_DIR
    / "development_and_holdout_performance.csv",
    index=False
)

In [ ]:
holdout_predictions = pd.DataFrame({
    "SK_ID_CURR": (
        id_hold.to_numpy()
    ),

    "TARGET_TRUE": (
        y_hold.to_numpy()
    ),

    "XGBOOST_PROBABILITY": (
        xgb_holdout_probability
    ),

    "LIGHTGBM_PROBABILITY": (
        lightgbm_holdout_probability
    ),

    "CATBOOST_PROBABILITY": (
        catboost_holdout_probability
    ),

    "ENSEMBLE_PROBABILITY": (
        final_holdout_probability
    )
})


assert holdout_predictions.shape[0] == (
    len(y_hold)
)

assert holdout_predictions[
    "SK_ID_CURR"
].is_unique

assert holdout_predictions.isna().sum().sum() == 0


holdout_predictions.to_csv(
    OUTPUT_DIR
    / "final_holdout_predictions.csv",
    index=False
)

holdout_predictions.head()

In [ ]:
final_holdout_result = pd.DataFrame({
    "metric": [
        "Development OOF ROC AUC",
        "Development OOF average precision",
        "Final holdout ROC AUC",
        "Final holdout average precision"
    ],

    "value": [
        V3_SELECTION_RESULTS[
            "Final_ensemble_oof_auc"
        ],

        V3_SELECTION_RESULTS[
            "Final_ensemble_oof_ap"
        ],

        final_holdout_auc,
        final_holdout_ap
    ]
})


final_holdout_result.to_csv(
    OUTPUT_DIR
    / "final_ensemble_result.csv",
    index=False
)

final_holdout_result

In [ ]:
Actually slightly better ROC AUC and average OOF precision for the holdout set than the development set
(0.790 and 0.291, respectively), which is a great indicator that the model is very robust.

The final equal-weight ensemble achieved an out-of-fold ROC AUC of 0.7868 and average precision of 0.2781 on the development sample. When evaluated once on the untouched 20% holdout sample, it achieved a ROC AUC of 0.7895 and average precision of 0.2912. The close agreement between development and holdout discrimination, with no deterioration on the holdout set, indicates that the selected features and ensemble generalized well and showed no evidence of substantial validation overfitting.

| Requested item            | V4 source                      |
| ------------------------- | ------------------------------ |
| Missing-value summary     | Raw application data           |
| Target-distribution chart | Full training target           |
| Correlation heatmap       | Selected numerical variables   |
| Baseline-model AUC        | Logistic-regression baseline   |
| Cross-validation AUC      | V3 ensemble OOF AUC            |
| Out-of-sample AUC         | V4 untouched holdout           |
| ROC curve                 | Holdout ensemble probabilities |
| Precision-recall curve    | Holdout ensemble probabilities |
| Confusion matrix          | Holdout, fixed 0.5 threshold   |
| Feature-importance chart  | Final XGBoost component        |
| SHAP summary plot         | Final XGBoost component        |
| Model-comparison table    | V1/V3/V4 results               |
| Kaggle submission file    | Full-data ensemble predictions |

In [ ]:
Performance deliverables

This section records three distinct stages of model performance:

1. Baseline performance from the earlier application-only models.
2. Cross-validated out-of-fold performance used for final model selection.
3. Final out-of-sample performance on the untouched holdout set.

The development OOF estimates were used for model selection. The holdout
metrics were calculated once after the pipeline was locked.

In [ ]:
# Replace np.nan with the saved V1 results.
#
# Keep the description accurate if the older models used a
# different feature set or validation procedure.

LEGACY_MODEL_RESULTS = {
    "Logistic regression": {
        "roc_auc": 0.629460950912641,

        # Average precision was not calculated in V1.
        "average_precision": np.nan,

        "feature_version": (
            "V1 application-only features; "
            "median/mode imputation and one-hot encoding"
        ),

        "evaluation_set": (
            "Single 80/20 validation split; "
            "logistic regression did not fully converge"
        )
    },

    "Random forest": {
        "roc_auc": 0.7107978692119733,

        # Average precision was not calculated in V1.
        "average_precision": np.nan,

        "feature_version": (
            "V1 application-only features; "
            "median/mode imputation and one-hot encoding"
        ),

        "evaluation_set": (
            "Single 80/20 validation split"
        )
    }
}

In [ ]:
for model_name, result in (
    LEGACY_MODEL_RESULTS.items()
):
    if pd.isna(result["roc_auc"]):
        print(
            f"Reminder: enter the saved "
            f"{model_name} ROC AUC."
        )

In [ ]:
baseline_performance_table = pd.DataFrame([
    {
        "model": model_name,
        "roc_auc": result["roc_auc"],
        "average_precision": (
            result["average_precision"]
        ),
        "feature_version": (
            result["feature_version"]
        ),
        "evaluation_set": (
            result["evaluation_set"]
        )
    }

    for model_name, result
    in LEGACY_MODEL_RESULTS.items()
])

baseline_performance_table.to_csv(
    OUTPUT_DIR
    / "baseline_model_performance.csv",
    index=False
)

baseline_performance_table

In [ ]:
performance_stage_table = pd.DataFrame({
    "performance_stage": [
        "Baseline model",
        "Development cross-validation",
        "Untouched holdout"
    ],

    "model_or_pipeline": [
        "Logistic regression",
        "Equal-weight ensemble",
        "Equal-weight ensemble"
    ],

    "roc_auc": [
        LEGACY_MODEL_RESULTS[
            "Logistic regression"
        ]["roc_auc"],

        V3_SELECTION_RESULTS[
            "Final_ensemble_oof_auc"
        ],

        final_holdout_auc
    ],

    "average_precision": [
        LEGACY_MODEL_RESULTS[
            "Logistic regression"
        ]["average_precision"],

        V3_SELECTION_RESULTS[
            "Final_ensemble_oof_ap"
        ],

        final_holdout_ap
    ],

    "interpretation": [
        (
            "Earlier application-only "
            "baseline"
        ),
        (
            "Five-fold out-of-fold "
            "development estimate"
        ),
        (
            "One-time final out-of-sample "
            "estimate"
        )
    ]
})

performance_stage_table.to_csv(
    OUTPUT_DIR
    / "performance_stage_summary.csv",
    index=False
)

performance_stage_table

In [ ]:
final_metric_summary = pd.Series({
    "Development OOF ROC AUC": (
        V3_SELECTION_RESULTS[
            "Final_ensemble_oof_auc"
        ]
    ),

    "Development OOF average precision": (
        V3_SELECTION_RESULTS[
            "Final_ensemble_oof_ap"
        ]
    ),

    "Final holdout ROC AUC": (
        final_holdout_auc
    ),

    "Final holdout average precision": (
        final_holdout_ap
    ),

    "Holdout default rate": (
        y_hold.mean()
    )
})

final_metric_summary

Final diagnostic plots

The ROC and precision-recall curves use the untouched holdout predictions
generated by the locked models.

The confusion matrix uses a fixed probability threshold of 0.50. This
threshold was not selected or optimized using the holdout labels.

In [ ]:
roc_curve_records = {}

for candidate_name, probability in (
    holdout_candidates.items()
):
    false_positive_rate, true_positive_rate, _ = (
        roc_curve(
            y_hold,
            probability
        )
    )

    roc_curve_records[
        candidate_name
    ] = {
        "false_positive_rate": (
            false_positive_rate
        ),
        "true_positive_rate": (
            true_positive_rate
        ),
        "auc": roc_auc_score(
            y_hold,
            probability
        )
    }

In [ ]:
figure, axis = plt.subplots(
    figsize=(8, 7)
)

for candidate_name, record in (
    roc_curve_records.items()
):
    line_width = (
        3
        if candidate_name
        == "Equal-weight ensemble"
        else 1.5
    )

    axis.plot(
        record["false_positive_rate"],
        record["true_positive_rate"],
        linewidth=line_width,
        label=(
            f"{candidate_name} "
            f"(AUC = {record['auc']:.4f})"
        )
    )

axis.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    linewidth=1,
    label="Random ranking"
)

axis.set_xlabel(
    "False positive rate"
)

axis.set_ylabel(
    "True positive rate"
)

axis.set_title(
    "ROC curves on the untouched holdout set"
)

axis.legend(
    loc="lower right"
)

axis.grid(
    alpha=0.25
)

figure.tight_layout()

figure.savefig(
    OUTPUT_DIR
    / "holdout_roc_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(figure)

In [ ]:
precision_recall_records = {}

for candidate_name, probability in (
    holdout_candidates.items()
):
    precision, recall, _ = (
        precision_recall_curve(
            y_hold,
            probability
        )
    )

    precision_recall_records[
        candidate_name
    ] = {
        "precision": precision,
        "recall": recall,
        "average_precision": (
            average_precision_score(
                y_hold,
                probability
            )
        )
    }

In [ ]:
holdout_prevalence = (
    y_hold.mean()
)

figure, axis = plt.subplots(
    figsize=(8, 7)
)

for candidate_name, record in (
    precision_recall_records.items()
):
    line_width = (
        3
        if candidate_name
        == "Equal-weight ensemble"
        else 1.5
    )

    axis.plot(
        record["recall"],
        record["precision"],
        linewidth=line_width,
        label=(
            f"{candidate_name} "
            f"(AP = "
            f"{record['average_precision']:.4f})"
        )
    )

axis.axhline(
    holdout_prevalence,
    linestyle="--",
    linewidth=1,
    label=(
        "Random-ranking baseline "
        f"({holdout_prevalence:.4f})"
    )
)

axis.set_xlabel(
    "Recall"
)

axis.set_ylabel(
    "Precision"
)

axis.set_title(
    "Precision-recall curves on the "
    "untouched holdout set"
)

axis.legend(
    loc="upper right"
)

axis.grid(
    alpha=0.25
)

figure.tight_layout()

figure.savefig(
    OUTPUT_DIR
    / "holdout_precision_recall_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(figure)

In [ ]:
CONFUSION_MATRIX_THRESHOLD = 0.50

ensemble_holdout_class = (
    final_holdout_probability
    >= CONFUSION_MATRIX_THRESHOLD
).astype("int8")

In [ ]:
ensemble_confusion_matrix = (
    confusion_matrix(
        y_hold,
        ensemble_holdout_class,
        labels=[0, 1]
    )
)

confusion_matrix_table = pd.DataFrame(
    ensemble_confusion_matrix,
    index=[
        "Actual non-default",
        "Actual default"
    ],
    columns=[
        "Predicted non-default",
        "Predicted default"
    ]
)

confusion_matrix_table.to_csv(
    OUTPUT_DIR
    / "holdout_confusion_matrix_counts.csv"
)

confusion_matrix_table

In [ ]:
figure, axis = plt.subplots(
    figsize=(7, 6)
)

display_object = ConfusionMatrixDisplay(
    confusion_matrix=(
        ensemble_confusion_matrix
    ),
    display_labels=[
        "No default",
        "Default"
    ]
)

display_object.plot(
    ax=axis,
    values_format=",d"
)

axis.set_title(
    "Final ensemble confusion matrix\n"
    f"Untouched holdout, threshold = "
    f"{CONFUSION_MATRIX_THRESHOLD:.2f}"
)

figure.tight_layout()

figure.savefig(
    OUTPUT_DIR
    / "holdout_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(figure)

In [ ]:
classification_report_table = (
    pd.DataFrame(
        classification_report(
            y_hold,
            ensemble_holdout_class,
            target_names=[
                "No default",
                "Default"
            ],
            output_dict=True,
            zero_division=0
        )
    )
    .transpose()
)

classification_report_table.to_csv(
    OUTPUT_DIR
    / "holdout_classification_report.csv"
)

classification_report_table

At the conventional probability threshold of 0.50, the ensemble classified relatively few applicants as defaults. This produced high precision among predicted defaults but very low default recall. The result does not conflict with the model’s strong ROC AUC and average precision, because those metrics assess probability ranking across all thresholds. A practical classification threshold would need to be selected according to the relative costs of missed defaults and false-positive flags.

In [ ]:
from sklearn.model_selection import StratifiedKFold


threshold_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)


xgb_oof_probability = np.zeros(
    len(y_dev),
    dtype="float64"
)

lightgbm_oof_probability = np.zeros(
    len(y_dev),
    dtype="float64"
)

catboost_oof_probability = np.zeros(
    len(y_dev),
    dtype="float64"
)

In [ ]:
XGB_OOF_SETTINGS = {
    **XGB_FINAL_SETTINGS,

    # Allow early stopping to select the fold-specific
    # iteration count.
    "n_estimators": 3000,
    "early_stopping_rounds": 100
}


LIGHTGBM_OOF_SETTINGS = {
    **LIGHTGBM_FINAL_SETTINGS,
    "n_estimators": 3000
}


CATBOOST_OOF_SETTINGS = {
    **CATBOOST_FINAL_SETTINGS,
    "iterations": 3000
}

In [ ]:
oof_fold_records = []


for fold_number, (
    training_positions,
    validation_positions
) in enumerate(
    threshold_cv.split(
        X_dev_xgb_final,
        y_dev
    ),
    start=1
):

    print(
        "\n" + "=" * 60
    )

    print(
        f"OOF fold {fold_number} of 5"
    )

    print(
        "=" * 60
    )


    y_fold_training = y_dev.iloc[
        training_positions
    ]

    y_fold_validation = y_dev.iloc[
        validation_positions
    ]


    # =====================================================
    # XGBoost
    # =====================================================

    xgb_fold_training = (
        X_dev_xgb_final.iloc[
            training_positions
        ]
    )

    xgb_fold_validation = (
        X_dev_xgb_final.iloc[
            validation_positions
        ]
    )


    xgb_fold_preprocessor = (
        build_xgb_preprocessor(
            selected_features=(
                X_dev_xgb_final.columns
            ),
            categorical_features=(
                categorical_features
            )
        )
    )


    xgb_fold_training_matrix = (
        xgb_fold_preprocessor
        .fit_transform(
            xgb_fold_training
        )
    )

    xgb_fold_validation_matrix = (
        xgb_fold_preprocessor
        .transform(
            xgb_fold_validation
        )
    )


    xgb_fold_model = XGBClassifier(
        **XGB_OOF_SETTINGS
    )


    xgb_fold_model.fit(
        xgb_fold_training_matrix,
        y_fold_training,

        eval_set=[
            (
                xgb_fold_validation_matrix,
                y_fold_validation
            )
        ],

        verbose=False
    )


    xgb_fold_probability = (
        xgb_fold_model.predict_proba(
            xgb_fold_validation_matrix
        )[:, 1]
    )


    xgb_oof_probability[
        validation_positions
    ] = xgb_fold_probability


    # =====================================================
    # CatBoost
    # =====================================================

    catboost_fold_training = (
        X_dev_catboost_final.iloc[
            training_positions
        ]
    )

    catboost_fold_validation = (
        X_dev_catboost_final.iloc[
            validation_positions
        ]
    )


    (
        catboost_fold_training_prepared,
        catboost_fold_validation_prepared,
        catboost_fold_categorical
    ) = prepare_catboost_pair(
        training_frame=(
            catboost_fold_training
        ),
        prediction_frame=(
            catboost_fold_validation
        ),
        categorical_features=(
            categorical_features
        )
    )


    catboost_fold_model = (
        CatBoostClassifier(
            **CATBOOST_OOF_SETTINGS
        )
    )


    catboost_fold_model.fit(
        catboost_fold_training_prepared,
        y_fold_training,

        cat_features=(
            catboost_fold_categorical
        ),

        eval_set=(
            catboost_fold_validation_prepared,
            y_fold_validation
        ),

        early_stopping_rounds=100,
        use_best_model=True,
        verbose=False
    )


    catboost_fold_probability = (
        catboost_fold_model.predict_proba(
            catboost_fold_validation_prepared
        )[:, 1]
    )


    catboost_oof_probability[
        validation_positions
    ] = catboost_fold_probability


    # =====================================================
    # LightGBM
    # =====================================================

    lightgbm_fold_training = (
        X_dev_lightgbm_final.iloc[
            training_positions
        ]
    )

    lightgbm_fold_validation = (
        X_dev_lightgbm_final.iloc[
            validation_positions
        ]
    )


    (
        lightgbm_fold_training_prepared,
        lightgbm_fold_validation_prepared,
        lightgbm_fold_categorical
    ) = prepare_lightgbm_pair(
        training_frame=(
            lightgbm_fold_training
        ),
        prediction_frame=(
            lightgbm_fold_validation
        ),
        categorical_features=(
            categorical_features
        )
    )


    lightgbm_fold_model = (
        LGBMClassifier(
            **LIGHTGBM_OOF_SETTINGS
        )
    )


    lightgbm_fold_model.fit(
        lightgbm_fold_training_prepared,
        y_fold_training,

        eval_set=[
            (
                lightgbm_fold_validation_prepared,
                y_fold_validation
            )
        ],

        eval_metric="auc",

        categorical_feature=(
            lightgbm_fold_categorical
        ),

        callbacks=[
            lgb.early_stopping(
                stopping_rounds=100,
                verbose=False
            )
        ]
    )


    lightgbm_fold_probability = (
        lightgbm_fold_model.predict_proba(
            lightgbm_fold_validation_prepared
        )[:, 1]
    )


    lightgbm_oof_probability[
        validation_positions
    ] = lightgbm_fold_probability


    # =====================================================
    # Fold metrics
    # =====================================================

    fold_ensemble_probability = (
        xgb_fold_probability
        + lightgbm_fold_probability
        + catboost_fold_probability
    ) / 3


    fold_record = {
        "fold": fold_number,

        "xgboost_auc": roc_auc_score(
            y_fold_validation,
            xgb_fold_probability
        ),

        "lightgbm_auc": roc_auc_score(
            y_fold_validation,
            lightgbm_fold_probability
        ),

        "catboost_auc": roc_auc_score(
            y_fold_validation,
            catboost_fold_probability
        ),

        "ensemble_auc": roc_auc_score(
            y_fold_validation,
            fold_ensemble_probability
        ),

        "ensemble_ap": (
            average_precision_score(
                y_fold_validation,
                fold_ensemble_probability
            )
        )
    }


    oof_fold_records.append(
        fold_record
    )


    print(
        "Fold ensemble AUC:",
        f"{fold_record['ensemble_auc']:.6f}"
    )

    print(
        "Fold ensemble AP:",
        f"{fold_record['ensemble_ap']:.6f}"
    )


    del (
        xgb_fold_preprocessor,
        xgb_fold_training_matrix,
        xgb_fold_validation_matrix,
        xgb_fold_model,
        catboost_fold_model,
        lightgbm_fold_model
    )

    gc.collect()

In [ ]:
final_oof_probability = (
    xgb_oof_probability
    + lightgbm_oof_probability
    + catboost_oof_probability
) / 3

In [ ]:
for model_name, probability in {
    "XGBoost": xgb_oof_probability,
    "LightGBM": lightgbm_oof_probability,
    "CatBoost": catboost_oof_probability,
    "Ensemble": final_oof_probability
}.items():

    assert len(probability) == len(
        y_dev
    )

    assert np.isfinite(
        probability
    ).all()

    assert probability.min() >= 0
    assert probability.max() <= 1

    print(
        model_name,
        "AUC:",
        f"{roc_auc_score(y_dev, probability):.6f}",
        "AP:",
        (
            f"{average_precision_score(
                y_dev,
                probability
            ):.6f}"
        )
    )

In [ ]:
development_oof_predictions = pd.DataFrame({
    "SK_ID_CURR": (
        id_dev.to_numpy()
    ),

    "TARGET_TRUE": (
        y_dev.to_numpy()
    ),

    "XGBOOST_OOF_PROBABILITY": (
        xgb_oof_probability
    ),

    "LIGHTGBM_OOF_PROBABILITY": (
        lightgbm_oof_probability
    ),

    "CATBOOST_OOF_PROBABILITY": (
        catboost_oof_probability
    ),

    "ENSEMBLE_OOF_PROBABILITY": (
        final_oof_probability
    )
})


assert development_oof_predictions[
    "SK_ID_CURR"
].is_unique

assert (
    development_oof_predictions
    .isna()
    .sum()
    .sum()
    == 0
)


development_oof_predictions.to_csv(
    OUTPUT_DIR
    / "development_oof_predictions.csv",
    index=False
)


development_oof_predictions.head()

In [ ]:
oof_fold_performance = pd.DataFrame(
    oof_fold_records
)

oof_fold_performance.to_csv(
    OUTPUT_DIR
    / "threshold_oof_fold_performance.csv",
    index=False
)

oof_fold_performance

In [ ]:
development_oof_predictions = pd.DataFrame({
    "TARGET_TRUE": y_dev.to_numpy(),

    "XGBOOST_OOF_PROBABILITY": (
        xgb_oof_probability
    ),

    "LIGHTGBM_OOF_PROBABILITY": (
        lightgbm_oof_probability
    ),

    "CATBOOST_OOF_PROBABILITY": (
        catboost_oof_probability
    ),

    "ENSEMBLE_OOF_PROBABILITY": (
        final_oof_probability
    )
})

development_oof_predictions.to_csv(
    OUTPUT_DIR
    / "development_oof_predictions.csv",
    index=False
)

In [ ]:
#threshold evaluation helper
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score
)


def evaluate_classification_threshold(
    y_true,
    probability,
    threshold
):
    """
    Calculate classification metrics at one probability threshold.
    """
    y_true_array = np.asarray(
        y_true
    )

    probability_array = np.asarray(
        probability
    )

    predicted_class = (
        probability_array >= threshold
    ).astype("int8")

    true_negative, false_positive, false_negative, true_positive = (
        confusion_matrix(
            y_true_array,
            predicted_class,
            labels=[0, 1]
        ).ravel()
    )

    precision = precision_score(
        y_true_array,
        predicted_class,
        zero_division=0
    )

    recall = recall_score(
        y_true_array,
        predicted_class,
        zero_division=0
    )

    f1 = f1_score(
        y_true_array,
        predicted_class,
        zero_division=0
    )

    beta = 2

    if (
        precision == 0
        and recall == 0
    ):
        f2 = 0.0
    else:
        f2 = (
            (1 + beta**2)
            * precision
            * recall
            / (
                beta**2
                * precision
                + recall
            )
        )

    specificity = (
        true_negative
        / (
            true_negative
            + false_positive
        )
    )

    return {
        "threshold": threshold,

        "predicted_default_count": int(
            predicted_class.sum()
        ),

        "predicted_default_rate": (
            predicted_class.mean()
        ),

        "true_negative": int(
            true_negative
        ),

        "false_positive": int(
            false_positive
        ),

        "false_negative": int(
            false_negative
        ),

        "true_positive": int(
            true_positive
        ),

        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1,
        "f2": f2,

        "balanced_accuracy": (
            balanced_accuracy_score(
                y_true_array,
                predicted_class
            )
        )
    }

In [ ]:
from sklearn.metrics import (
    precision_recall_curve
)


oof_precision, oof_recall, oof_thresholds = (
    precision_recall_curve(
        y_dev,
        final_oof_probability
    )
)


threshold_curve = pd.DataFrame({
    "threshold": oof_thresholds,

    # Precision and recall contain one extra endpoint.
    "precision": oof_precision[:-1],
    "recall": oof_recall[:-1]
})

In [ ]:
threshold_curve[
    "f1"
] = np.divide(
    2
    * threshold_curve["precision"]
    * threshold_curve["recall"],

    (
        threshold_curve["precision"]
        + threshold_curve["recall"]
    ),

    out=np.zeros(
        len(threshold_curve)
    ),

    where=(
        threshold_curve["precision"]
        + threshold_curve["recall"]
    ) > 0
)


beta = 2

threshold_curve[
    "f2"
] = np.divide(
    (
        1
        + beta**2
    )
    * threshold_curve["precision"]
    * threshold_curve["recall"],

    (
        beta**2
        * threshold_curve["precision"]
        + threshold_curve["recall"]
    ),

    out=np.zeros(
        len(threshold_curve)
    ),

    where=(
        beta**2
        * threshold_curve["precision"]
        + threshold_curve["recall"]
    ) > 0
)

In [ ]:
threshold_curve.to_csv(
    OUTPUT_DIR
    / "development_oof_threshold_curve.csv",
    index=False
)

In [ ]:
maximum_f1_row = threshold_curve.loc[
    threshold_curve["f1"].idxmax()
]

maximum_f1_threshold = float(
    maximum_f1_row["threshold"]
)

maximum_f1_row

In [ ]:
maximum_f2_row = threshold_curve.loc[
    threshold_curve["f2"].idxmax()
]

maximum_f2_threshold = float(
    maximum_f2_row["threshold"]
)

maximum_f2_row

In [ ]:
def choose_threshold_for_target_recall(
    threshold_table,
    target_recall
):
    """
    Among thresholds meeting the requested recall, select the
    threshold with the highest precision.
    """
    eligible = threshold_table.loc[
        threshold_table["recall"]
        >= target_recall
    ].copy()

    if eligible.empty:
        raise ValueError(
            "No threshold reaches recall "
            f"{target_recall:.1%}."
        )

    return (
        eligible
        .sort_values(
            [
                "precision",
                "threshold"
            ],
            ascending=[
                False,
                False
            ]
        )
        .iloc[0]
    )

In [ ]:
recall_50_row = (
    choose_threshold_for_target_recall(
        threshold_curve,
        target_recall=0.50
    )
)

recall_60_row = (
    choose_threshold_for_target_recall(
        threshold_curve,
        target_recall=0.60
    )
)

recall_70_row = (
    choose_threshold_for_target_recall(
        threshold_curve,
        target_recall=0.70
    )
)

In [ ]:
candidate_thresholds = {
    "Conventional 0.50": 0.50,

    "Maximum OOF F1": (
        maximum_f1_threshold
    ),

    "Maximum OOF F2": (
        maximum_f2_threshold
    ),

    "At least 50% OOF recall": float(
        recall_50_row["threshold"]
    ),

    "At least 60% OOF recall": float(
        recall_60_row["threshold"]
    ),

    "At least 70% OOF recall": float(
        recall_70_row["threshold"]
    )
}

In [ ]:
candidate_threshold_table = pd.DataFrame([
    {
        "selection_rule": rule,
        **evaluate_classification_threshold(
            y_true=y_dev,
            probability=(
                final_oof_probability
            ),
            threshold=threshold
        )
    }

    for rule, threshold
    in candidate_thresholds.items()
])


candidate_threshold_table = (
    candidate_threshold_table
    .sort_values(
        "threshold",
        ascending=False
    )
    .reset_index(drop=True)
)

candidate_threshold_table.to_csv(
    OUTPUT_DIR
    / "development_oof_candidate_thresholds.csv",
    index=False
)

candidate_threshold_table

In [ ]:
# A light downsample keeps the plot manageable if the
# threshold curve has hundreds of thousands of rows.

plot_step = max(
    1,
    len(threshold_curve) // 3000
)

threshold_curve_plot = (
    threshold_curve
    .iloc[::plot_step]
)

In [ ]:
figure, axis = plt.subplots(
    figsize=(10, 7)
)

axis.plot(
    threshold_curve_plot[
        "threshold"
    ],
    threshold_curve_plot[
        "precision"
    ],
    label="Precision"
)

axis.plot(
    threshold_curve_plot[
        "threshold"
    ],
    threshold_curve_plot[
        "recall"
    ],
    label="Recall"
)

axis.plot(
    threshold_curve_plot[
        "threshold"
    ],
    threshold_curve_plot[
        "f1"
    ],
    label="F1"
)

axis.plot(
    threshold_curve_plot[
        "threshold"
    ],
    threshold_curve_plot[
        "f2"
    ],
    label="F2"
)

axis.axvline(
    maximum_f2_threshold,
    linestyle="--",
    linewidth=1.5,
    label=(
        "Maximum-F2 threshold "
        f"({maximum_f2_threshold:.4f})"
    )
)

axis.set_xlabel(
    "Classification threshold"
)

axis.set_ylabel(
    "Metric value"
)

axis.set_title(
    "Development OOF threshold analysis"
)

axis.legend()

axis.grid(
    alpha=0.25
)

figure.tight_layout()

figure.savefig(
    OUTPUT_DIR
    / "development_oof_threshold_analysis.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(figure)

In [ ]:
LOCKED_CLASSIFICATION_THRESHOLD = maximum_f2_threshold

THRESHOLD_SELECTION_RULE = (
    "Threshold maximizing F2 on development "
    "out-of-fold ensemble predictions"
)

At threshold ~= 0.21, precision and recall are approximately equal. 

The conventional threshold of 0.50 produced high precision but identified only 4.3% of defaults in development OOF predictions. Because missed defaults were considered more consequential than additional false-positive flags, the final classification threshold was selected by maximizing the F2 score on development OOF probabilities. This produced a threshold of 0.0878, with development recall of 68.2%, precision of 19.2%, and specificity of 74.8%. Because the model is interpreted as a risk-screening system rather than an automatic rejection rule, missed defaults were considered more consequential than additional false-positive flags.

In [ ]:
CONFUSION_MATRIX_THRESHOLD = 0.087754	

ensemble_holdout_class = (
    final_holdout_probability
    >= CONFUSION_MATRIX_THRESHOLD
).astype("int8")

In [ ]:
oof_prediction_path = (
    OUTPUT_DIR
    / "development_oof_predictions.csv"
)


if "final_oof_probability" not in globals():
    assert oof_prediction_path.exists(), (
        "The OOF prediction file is missing. "
        "Regenerate and save the final five-fold OOF predictions first."
    )

    development_oof_predictions = pd.read_csv(
        oof_prediction_path
    )

    assert np.array_equal(
        development_oof_predictions[
            "SK_ID_CURR"
        ].to_numpy(),
        id_dev.to_numpy()
    )

    assert np.array_equal(
        development_oof_predictions[
            "TARGET_TRUE"
        ].to_numpy(),
        y_dev.to_numpy()
    )

    xgb_oof_probability = (
        development_oof_predictions[
            "XGBOOST_OOF_PROBABILITY"
        ].to_numpy()
    )

    lightgbm_oof_probability = (
        development_oof_predictions[
            "LIGHTGBM_OOF_PROBABILITY"
        ].to_numpy()
    )

    catboost_oof_probability = (
        development_oof_predictions[
            "CATBOOST_OOF_PROBABILITY"
        ].to_numpy()
    )

    final_oof_probability = (
        development_oof_predictions[
            "ENSEMBLE_OOF_PROBABILITY"
        ].to_numpy()
    )


print(
    "OOF observations:",
    f"{len(final_oof_probability):,}"
)

print(
    "OOF ensemble ROC AUC:",
    f"{roc_auc_score(y_dev, final_oof_probability):.6f}"
)

print(
    "OOF ensemble average precision:",
    (
        f"{average_precision_score(
            y_dev,
            final_oof_probability
        ):.6f}"
    )
)

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    balanced_accuracy_score
)

In [ ]:
def evaluate_classification_threshold(
    y_true,
    probability,
    threshold
):
    """
    Evaluate binary classifications created at one probability
    threshold.
    """
    y_true_array = np.asarray(
        y_true
    )

    probability_array = np.asarray(
        probability
    )

    predicted_class = (
        probability_array
        >= threshold
    ).astype("int8")

    (
        true_negative,
        false_positive,
        false_negative,
        true_positive
    ) = confusion_matrix(
        y_true_array,
        predicted_class,
        labels=[0, 1]
    ).ravel()

    precision = precision_score(
        y_true_array,
        predicted_class,
        zero_division=0
    )

    recall = recall_score(
        y_true_array,
        predicted_class,
        zero_division=0
    )

    specificity = (
        true_negative
        / (
            true_negative
            + false_positive
        )
    )

    return {
        "threshold": float(
            threshold
        ),

        "predicted_default_count": int(
            predicted_class.sum()
        ),

        "predicted_default_rate": float(
            predicted_class.mean()
        ),

        "true_negative": int(
            true_negative
        ),

        "false_positive": int(
            false_positive
        ),

        "false_negative": int(
            false_negative
        ),

        "true_positive": int(
            true_positive
        ),

        "precision": float(
            precision
        ),

        "recall": float(
            recall
        ),

        "specificity": float(
            specificity
        ),

        "f1": float(
            f1_score(
                y_true_array,
                predicted_class,
                zero_division=0
            )
        ),

        "f2": float(
            fbeta_score(
                y_true_array,
                predicted_class,
                beta=2,
                zero_division=0
            )
        ),

        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true_array,
                predicted_class
            )
        )
    }

In [ ]:
oof_precision, oof_recall, oof_thresholds = (
    precision_recall_curve(
        y_dev,
        final_oof_probability
    )
)


threshold_curve = pd.DataFrame({
    "threshold": (
        oof_thresholds
    ),

    # Precision and recall contain one extra endpoint.
    "precision": (
        oof_precision[:-1]
    ),

    "recall": (
        oof_recall[:-1]
    )
})

In [ ]:
threshold_curve[
    "f1"
] = np.divide(
    2
    * threshold_curve["precision"]
    * threshold_curve["recall"],

    (
        threshold_curve["precision"]
        + threshold_curve["recall"]
    ),

    out=np.zeros(
        len(threshold_curve)
    ),

    where=(
        threshold_curve["precision"]
        + threshold_curve["recall"]
    ) > 0
)

In [ ]:
F2_BETA = 2


threshold_curve[
    "f2"
] = np.divide(
    (
        1
        + F2_BETA**2
    )
    * threshold_curve["precision"]
    * threshold_curve["recall"],

    (
        F2_BETA**2
        * threshold_curve["precision"]
        + threshold_curve["recall"]
    ),

    out=np.zeros(
        len(threshold_curve)
    ),

    where=(
        F2_BETA**2
        * threshold_curve["precision"]
        + threshold_curve["recall"]
    ) > 0
)


threshold_curve.to_csv(
    OUTPUT_DIR
    / "development_oof_threshold_curve.csv",
    index=False
)

In [ ]:
maximum_f1_row = threshold_curve.loc[
    threshold_curve[
        "f1"
    ].idxmax()
]

maximum_f2_row = threshold_curve.loc[
    threshold_curve[
        "f2"
    ].idxmax()
]


maximum_f1_threshold = float(
    maximum_f1_row[
        "threshold"
    ]
)

maximum_f2_threshold = float(
    maximum_f2_row[
        "threshold"
    ]
)


print(
    "Maximum-F1 threshold:",
    f"{maximum_f1_threshold:.6f}"
)

print(
    "Maximum-F2 threshold:",
    f"{maximum_f2_threshold:.6f}"
)

In [ ]:
LOCKED_CLASSIFICATION_THRESHOLD = (
    maximum_f2_threshold
)

THRESHOLD_SELECTION_RULE = (
    "Threshold maximizing F2 on development "
    "out-of-fold ensemble predictions"
)


print(
    "Locked classification threshold:",
    f"{LOCKED_CLASSIFICATION_THRESHOLD:.6f}"
)

print(
    "Selection rule:",
    THRESHOLD_SELECTION_RULE
)

In [ ]:
locked_development_threshold_result = (
    evaluate_classification_threshold(
        y_true=y_dev,
        probability=(
            final_oof_probability
        ),
        threshold=(
            LOCKED_CLASSIFICATION_THRESHOLD
        )
    )
)


locked_development_threshold_table = (
    pd.DataFrame([
        {
            "dataset": (
                "Development OOF"
            ),

            "selection_rule": (
                THRESHOLD_SELECTION_RULE
            ),

            **locked_development_threshold_result
        }
    ])
)


locked_development_threshold_table.to_csv(
    OUTPUT_DIR
    / "locked_f2_threshold_development_metrics.csv",
    index=False
)


locked_development_threshold_table

Holdout evaluation at the locked operating threshold

The threshold was selected entirely from development out-of-fold
predictions. It is now applied once to the untouched holdout probabilities.

The threshold changes only hard class assignments. It does not change ROC
AUC, average precision, model parameters, ensemble weights, or Kaggle
probability predictions.

In [ ]:
locked_holdout_class = (
    final_holdout_probability
    >= LOCKED_CLASSIFICATION_THRESHOLD
).astype("int8")

In [ ]:
locked_holdout_threshold_result = (
    evaluate_classification_threshold(
        y_true=y_hold,
        probability=(
            final_holdout_probability
        ),
        threshold=(
            LOCKED_CLASSIFICATION_THRESHOLD
        )
    )
)


locked_holdout_threshold_table = (
    pd.DataFrame([
        {
            "dataset": (
                "Untouched holdout"
            ),

            "selection_rule": (
                THRESHOLD_SELECTION_RULE
            ),

            **locked_holdout_threshold_result
        }
    ])
)


locked_holdout_threshold_table.to_csv(
    OUTPUT_DIR
    / "locked_f2_threshold_holdout_metrics.csv",
    index=False
)


locked_holdout_threshold_table

In [ ]:
locked_threshold_comparison = pd.concat(
    [
        locked_development_threshold_table,
        locked_holdout_threshold_table
    ],
    ignore_index=True
)


locked_threshold_comparison.to_csv(
    OUTPUT_DIR
    / "locked_f2_threshold_development_holdout_comparison.csv",
    index=False
)


locked_threshold_comparison

In [ ]:
locked_confusion_matrix = confusion_matrix(
    y_hold,
    locked_holdout_class,
    labels=[0, 1]
)


locked_confusion_matrix_table = pd.DataFrame(
    locked_confusion_matrix,
    index=[
        "Actual non-default",
        "Actual default"
    ],
    columns=[
        "Predicted non-default",
        "Predicted default"
    ]
)


locked_confusion_matrix_table.to_csv(
    OUTPUT_DIR
    / "holdout_confusion_matrix_locked_f2_counts.csv"
)


locked_confusion_matrix_table

In [ ]:
figure, axis = plt.subplots(
    figsize=(7, 6)
)


display_object = ConfusionMatrixDisplay(
    confusion_matrix=(
        locked_confusion_matrix
    ),
    display_labels=[
        "No default",
        "Default"
    ]
)


display_object.plot(
    ax=axis,
    values_format=",d"
)


axis.set_title(
    "Final ensemble confusion matrix\n"
    "OOF maximum-F2 threshold = "
    f"{LOCKED_CLASSIFICATION_THRESHOLD:.4f}"
)


figure.tight_layout()


figure.savefig(
    OUTPUT_DIR
    / "holdout_confusion_matrix_locked_f2.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()
plt.close(figure)

In [ ]:
conventional_holdout_result = (
    evaluate_classification_threshold(
        y_true=y_hold,
        probability=(
            final_holdout_probability
        ),
        threshold=0.50
    )
)


holdout_threshold_comparison = pd.DataFrame([
    {
        "threshold_rule": (
            "Conventional 0.50"
        ),

        **conventional_holdout_result
    },

    {
        "threshold_rule": (
            "Development OOF maximum F2"
        ),

        **locked_holdout_threshold_result
    }
])


holdout_threshold_comparison.to_csv(
    OUTPUT_DIR
    / "holdout_threshold_comparison.csv",
    index=False
)


holdout_threshold_comparison

Feature interpretation

The final predictive system is an ensemble and therefore does not have one
single feature-importance vector.

XGBoost is used as the representative interpretable component because it
was the strongest individual development model, retained the complete final
feature set, and supports gain-based importance and SHAP analysis.

These plots describe the XGBoost component rather than an exact importance
decomposition of the entire ensemble.

In [ ]:
if "xgb_processed_feature_names" not in globals():
    xgb_processed_feature_names = (
        xgb_preprocessor
        .get_feature_names_out()
    )


def clean_xgb_feature_name(
    feature_name
):
    cleaned = str(
        feature_name
    )

    for prefix in [
        "numeric__",
        "categorical__"
    ]:
        if cleaned.startswith(
            prefix
        ):
            cleaned = cleaned[
                len(prefix):
            ]

    return cleaned


xgb_clean_feature_names = np.array([
    clean_xgb_feature_name(
        feature_name
    )

    for feature_name in (
        xgb_processed_feature_names
    )
])

In [ ]:
xgb_feature_importance = pd.DataFrame({
    "feature": (
        xgb_clean_feature_names
    ),

    "importance": (
        xgb_final_model
        .feature_importances_
    )
})


assert len(
    xgb_feature_importance
) == len(
    xgb_clean_feature_names
)


xgb_feature_importance = (
    xgb_feature_importance
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(drop=True)
)


xgb_feature_importance.to_csv(
    OUTPUT_DIR
    / "xgboost_component_feature_importance.csv",
    index=False
)


xgb_feature_importance.head(25)

In [ ]:
top_xgb_importance = (
    xgb_feature_importance
    .head(25)
    .sort_values(
        "importance",
        ascending=True
    )
)


figure, axis = plt.subplots(
    figsize=(10, 9)
)


axis.barh(
    top_xgb_importance[
        "feature"
    ],
    top_xgb_importance[
        "importance"
    ]
)


axis.set_xlabel(
    "Gain-based feature importance"
)

axis.set_ylabel(
    "Feature"
)

axis.set_title(
    "XGBoost ensemble component:\n"
    "Top 25 feature importances"
)

axis.grid(
    axis="x",
    alpha=0.25
)


figure.tight_layout()


figure.savefig(
    OUTPUT_DIR
    / "xgboost_component_feature_importance.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()
plt.close(figure)

In [ ]:
if "xgb_holdout_matrix" not in globals():
    xgb_holdout_matrix = (
        xgb_preprocessor
        .transform(
            X_hold_xgb_final
        )
    )


SHAP_SAMPLE_SIZE = min(
    3000,
    len(y_hold)
)


shap_random_generator = (
    np.random.default_rng(
        RANDOM_STATE
    )
)


shap_sample_positions = (
    shap_random_generator.choice(
        len(y_hold),
        size=SHAP_SAMPLE_SIZE,
        replace=False
    )
)


xgb_shap_matrix = (
    xgb_holdout_matrix[
        shap_sample_positions
    ]
)


if hasattr(
    xgb_shap_matrix,
    "toarray"
):
    xgb_shap_matrix_dense = (
        xgb_shap_matrix.toarray()
    )
else:
    xgb_shap_matrix_dense = (
        np.asarray(
            xgb_shap_matrix
        )
    )


print(
    "SHAP sample shape:",
    xgb_shap_matrix_dense.shape
)

In [ ]:
xgb_shap_explainer = shap.TreeExplainer(
    xgb_final_model
)


try:
    shap_explanation = (
        xgb_shap_explainer(
            xgb_shap_matrix_dense,
            check_additivity=False
        )
    )

    xgb_shap_values = np.asarray(
        shap_explanation.values
    )

except Exception:
    xgb_shap_values = (
        xgb_shap_explainer
        .shap_values(
            xgb_shap_matrix_dense,
            check_additivity=False
        )
    )

    if isinstance(
        xgb_shap_values,
        list
    ):
        xgb_shap_values = (
            xgb_shap_values[-1]
        )

    xgb_shap_values = np.asarray(
        xgb_shap_values
    )


if xgb_shap_values.ndim == 3:
    xgb_shap_values = (
        xgb_shap_values[:, :, -1]
    )


assert xgb_shap_values.shape == (
    xgb_shap_matrix_dense.shape
)

In [ ]:
plt.figure(
    figsize=(11, 9)
)


shap.summary_plot(
    xgb_shap_values,
    xgb_shap_matrix_dense,
    feature_names=(
        xgb_clean_feature_names
    ),
    max_display=25,
    show=False
)


plt.title(
    "XGBoost ensemble component:\n"
    "SHAP summary plot"
)


plt.tight_layout()


plt.savefig(
    OUTPUT_DIR
    / "xgboost_component_shap_summary.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()
plt.close()

In [ ]:
xgb_mean_absolute_shap = pd.DataFrame({
    "feature": (
        xgb_clean_feature_names
    ),

    "mean_absolute_shap": (
        np.abs(
            xgb_shap_values
        ).mean(axis=0)
    )
})


xgb_mean_absolute_shap = (
    xgb_mean_absolute_shap
    .sort_values(
        "mean_absolute_shap",
        ascending=False
    )
    .reset_index(drop=True)
)


xgb_mean_absolute_shap.to_csv(
    OUTPUT_DIR
    / "xgboost_component_mean_absolute_shap.csv",
    index=False
)


xgb_mean_absolute_shap.head(25)

In [ ]:
top_shap_features = (
    xgb_mean_absolute_shap
    .head(25)
    .sort_values(
        "mean_absolute_shap",
        ascending=True
    )
)


figure, axis = plt.subplots(
    figsize=(10, 9)
)


axis.barh(
    top_shap_features[
        "feature"
    ],
    top_shap_features[
        "mean_absolute_shap"
    ]
)


axis.set_xlabel(
    "Mean absolute SHAP value"
)

axis.set_ylabel(
    "Feature"
)

axis.set_title(
    "XGBoost ensemble component:\n"
    "Top 25 features by mean absolute SHAP"
)

axis.grid(
    axis="x",
    alpha=0.25
)


figure.tight_layout()


figure.savefig(
    OUTPUT_DIR
    / "xgboost_component_shap_importance.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()
plt.close(figure)

In [ ]:
xgb_importance_comparison = (
    xgb_feature_importance
    .merge(
        xgb_mean_absolute_shap,
        on="feature",
        how="outer",
        validate="one_to_one"
    )
    .fillna(0)
)


xgb_importance_comparison[
    "gain_rank"
] = (
    xgb_importance_comparison[
        "importance"
    ]
    .rank(
        ascending=False,
        method="min"
    )
)


xgb_importance_comparison[
    "shap_rank"
] = (
    xgb_importance_comparison[
        "mean_absolute_shap"
    ]
    .rank(
        ascending=False,
        method="min"
    )
)


xgb_importance_comparison = (
    xgb_importance_comparison
    .sort_values(
        "shap_rank"
    )
    .reset_index(drop=True)
)


xgb_importance_comparison.to_csv(
    OUTPUT_DIR
    / "xgboost_importance_method_comparison.csv",
    index=False
)


xgb_importance_comparison.head(25)

Organization type (broader categorical variables) influences XGBoost predictions.

Gain importance identifies
EXT_SOURCE_2
EXT_SOURCE_3
Higher education
Gender
Bureau debt ratios
Credit/goods ratio
Employment duration
Installment late rate

Doesn't always agree with the importance for making splits variables, e.g.
ORGANIZATION_TYPE_Realtor
Gain rank: 159
SHAP rank: 1

A counterintuitive pattern is that high values of INST_UNDERPAID_COUNT are more on the negative side, which means more recorded underpayments lower predicted risk. 
Maybe it's because applicants with more installment records have more opportunities to accumulate an underpaid count. 

Still, external-source risk scores are major predictors, with high scores correlated with greater risk.
Bureau debt and installment payment behavior are meaningful historical information.
Financial obligation variables such as annuity-to-credit ratio affect risk.
Demographic and employment variables also help.

In [ ]:
# Reproduce the feature ordering used by the XGBoost preprocessor.

categorical_feature_set = set(
    categorical_features
)

xgb_categorical_features = [
    feature
    for feature in X_dev_xgb_final.columns
    if feature in categorical_feature_set
]

xgb_numerical_features = [
    feature
    for feature in X_dev_xgb_final.columns
    if feature not in categorical_feature_set
]


xgb_onehot_encoder = (
    xgb_preprocessor
    .named_transformers_[
        "categorical"
    ]
    .named_steps[
        "onehot"
    ]
)


processed_to_original_feature = []

# Numerical variables each correspond to one processed column.
processed_to_original_feature.extend(
    xgb_numerical_features
)

# Each one-hot column maps back to its original categorical variable.
for feature, categories in zip(
    xgb_categorical_features,
    xgb_onehot_encoder.categories_
):
    processed_to_original_feature.extend(
        [feature] * len(categories)
    )


processed_to_original_feature = np.array(
    processed_to_original_feature
)


assert len(
    processed_to_original_feature
) == xgb_shap_values.shape[1]

assert len(
    processed_to_original_feature
) == len(
    xgb_clean_feature_names
)

In [ ]:
grouped_shap_records = []


for original_feature in pd.unique(
    processed_to_original_feature
):
    feature_positions = np.flatnonzero(
        processed_to_original_feature
        == original_feature
    )

    # SHAP is additive, so sum dummy contributions within
    # each applicant before calculating average magnitude.
    grouped_contribution = (
        xgb_shap_values[
            :,
            feature_positions
        ]
        .sum(axis=1)
    )

    grouped_shap_records.append({
        "feature": original_feature,

        "number_of_processed_columns": int(
            len(feature_positions)
        ),

        "mean_absolute_grouped_shap": float(
            np.abs(
                grouped_contribution
            ).mean()
        )
    })


xgb_grouped_shap_importance = (
    pd.DataFrame(
        grouped_shap_records
    )
    .sort_values(
        "mean_absolute_grouped_shap",
        ascending=False
    )
    .reset_index(drop=True)
)


xgb_grouped_shap_importance.to_csv(
    OUTPUT_DIR
    / "xgboost_grouped_shap_importance.csv",
    index=False
)


xgb_grouped_shap_importance.head(25)

In [ ]:
top_grouped_shap = (
    xgb_grouped_shap_importance
    .head(25)
    .sort_values(
        "mean_absolute_grouped_shap",
        ascending=True
    )
)


figure, axis = plt.subplots(
    figsize=(10, 9)
)


axis.barh(
    top_grouped_shap[
        "feature"
    ],
    top_grouped_shap[
        "mean_absolute_grouped_shap"
    ]
)


axis.set_xlabel(
    "Mean absolute grouped SHAP value"
)

axis.set_ylabel(
    "Original feature"
)

axis.set_title(
    "XGBoost ensemble component:\n"
    "Top original features by grouped SHAP importance"
)

axis.grid(
    axis="x",
    alpha=0.25
)


figure.tight_layout()


figure.savefig(
    OUTPUT_DIR
    / "xgboost_component_grouped_shap_importance.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()
plt.close(figure)

In [ ]:
onehot_diagnostics = []


for feature_name in (
    xgb_mean_absolute_shap[
        "feature"
    ]
    .head(25)
):
    feature_positions = np.flatnonzero(
        xgb_clean_feature_names
        == feature_name
    )

    if len(feature_positions) != 1:
        continue

    position = feature_positions[0]

    feature_values = (
        xgb_shap_matrix_dense[
            :,
            position
        ]
    )

    unique_values = np.unique(
        feature_values[
            np.isfinite(
                feature_values
            )
        ]
    )

    # Restrict to binary one-hot columns.
    if not set(
        unique_values
    ).issubset({0, 1}):
        continue

    present = (
        feature_values == 1
    )

    absent = (
        feature_values == 0
    )

    onehot_diagnostics.append({
        "feature": feature_name,

        "sample_prevalence": float(
            present.mean()
        ),

        "number_present": int(
            present.sum()
        ),

        "mean_shap_when_present": float(
            xgb_shap_values[
                present,
                position
            ].mean()
        )
        if present.any()
        else np.nan,

        "mean_shap_when_absent": float(
            xgb_shap_values[
                absent,
                position
            ].mean()
        )
        if absent.any()
        else np.nan,

        "mean_absolute_shap": float(
            np.abs(
                xgb_shap_values[
                    :,
                    position
                ]
            ).mean()
        )
    })


onehot_diagnostic_table = (
    pd.DataFrame(
        onehot_diagnostics
    )
    .sort_values(
        "mean_absolute_shap",
        ascending=False
    )
    .reset_index(drop=True)
)


onehot_diagnostic_table

Suspicious:
ORGANIZATION_TYPE_Realtor
Present in only 4 of 3,000 observations

Mean SHAP when present:  0.390
Mean SHAP when absent:   0.403

FLAG_DOCUMENT_11
Mean SHAP when present: -0.247
Mean SHAP when absent:  -0.259

Same contribution whether present or not and pretty large contribution, too. Possible problem that sparse XGBoost input was converted to dense.

In [ ]:
sparse_shap_sample_probability = (
    xgb_final_model.predict_proba(
        xgb_shap_matrix
    )[:, 1]
)

dense_shap_sample_probability = (
    xgb_final_model.predict_proba(
        xgb_shap_matrix_dense
    )[:, 1]
)


prediction_representation_difference = (
    np.abs(
        sparse_shap_sample_probability
        - dense_shap_sample_probability
    )
)


pd.Series(
    prediction_representation_difference
).describe(
    percentiles=[
        0.50,
        0.90,
        0.95,
        0.99
    ]
)

In [ ]:
print(
    "Maximum probability difference:",
    prediction_representation_difference.max()
)

print(
    "Mean probability difference:",
    prediction_representation_difference.mean()
)

print(
    "Predictions effectively identical:",
    np.allclose(
        sparse_shap_sample_probability,
        dense_shap_sample_probability,
        atol=1e-8,
        rtol=1e-6
    )
)

OK, so there was definitely a problem with the representation.

In [ ]:
import xgboost as xgb


# Keep the exact sparse representation used by the fitted model.
xgb_shap_dmatrix = xgb.DMatrix(
    xgb_shap_matrix,
    feature_names=(
        xgb_clean_feature_names
        .astype(str)
        .tolist()
    )
)


xgb_native_contributions = (
    xgb_final_model
    .get_booster()
    .predict(
        xgb_shap_dmatrix,
        pred_contribs=True
    )
)


# The final column is the expected/base model output.
xgb_shap_values = (
    xgb_native_contributions[
        :,
        :-1
    ]
)

xgb_shap_base_values = (
    xgb_native_contributions[
        :,
        -1
    ]
)


assert xgb_shap_values.shape == (
    xgb_shap_matrix.shape[0],
    xgb_shap_matrix.shape[1]
)

In [ ]:
xgb_raw_margin = (
    xgb_final_model
    .get_booster()
    .predict(
        xgb_shap_dmatrix,
        output_margin=True
    )
)


reconstructed_raw_margin = (
    xgb_shap_base_values
    + xgb_shap_values.sum(axis=1)
)


reconstruction_error = np.abs(
    reconstructed_raw_margin
    - xgb_raw_margin
)


print(
    "SHAP reconstruction valid:",
    np.allclose(
        reconstructed_raw_margin,
        xgb_raw_margin,
        atol=1e-5,
        rtol=1e-5
    )
)

print(
    "Maximum raw-margin reconstruction error:",
    reconstruction_error.max()
)

In [ ]:
native_shap_probability = (
    1
    / (
        1
        + np.exp(
            -xgb_raw_margin
        )
    )
)


print(
    "Native SHAP probabilities match "
    "sparse predict_proba:",
    np.allclose(
        native_shap_probability,
        sparse_shap_sample_probability,
        atol=1e-6,
        rtol=1e-6
    )
)

print(
    "Maximum probability difference:",
    np.max(
        np.abs(
            native_shap_probability
            - sparse_shap_sample_probability
        )
    )
)

In [ ]:
xgb_shap_display_values = (
    xgb_shap_matrix.toarray()
    if hasattr(
        xgb_shap_matrix,
        "toarray"
    )
    else np.asarray(
        xgb_shap_matrix
    )
)

In [ ]:
plt.figure(
    figsize=(11, 9)
)


shap.summary_plot(
    xgb_shap_values,
    xgb_shap_display_values,
    feature_names=(
        xgb_clean_feature_names
    ),
    max_display=25,
    show=False
)


plt.title(
    "XGBoost ensemble component:\n"
    "SHAP summary plot"
)


plt.tight_layout()


plt.savefig(
    OUTPUT_DIR
    / "xgboost_component_shap_summary.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()
plt.close()

In [ ]:
xgb_mean_absolute_shap = pd.DataFrame({
    "feature": (
        xgb_clean_feature_names
    ),

    "mean_absolute_shap": (
        np.abs(
            xgb_shap_values
        ).mean(axis=0)
    )
})


xgb_mean_absolute_shap = (
    xgb_mean_absolute_shap
    .sort_values(
        "mean_absolute_shap",
        ascending=False
    )
    .reset_index(drop=True)
)


xgb_mean_absolute_shap.to_csv(
    OUTPUT_DIR
    / "xgboost_component_mean_absolute_shap.csv",
    index=False
)


xgb_mean_absolute_shap.head(25)

In [ ]:
top_shap_features = (
    xgb_mean_absolute_shap
    .head(25)
    .sort_values(
        "mean_absolute_shap",
        ascending=True
    )
)


figure, axis = plt.subplots(
    figsize=(10, 9)
)


axis.barh(
    top_shap_features[
        "feature"
    ],
    top_shap_features[
        "mean_absolute_shap"
    ]
)


axis.set_xlabel(
    "Mean absolute SHAP value"
)

axis.set_ylabel(
    "Feature"
)

axis.set_title(
    "XGBoost ensemble component:\n"
    "Top 25 features by mean absolute SHAP"
)

axis.grid(
    axis="x",
    alpha=0.25
)


figure.tight_layout()


figure.savefig(
    OUTPUT_DIR
    / "xgboost_component_shap_importance.png",
    dpi=300,
    bbox_inches="tight"
)


plt.show()
plt.close(figure)

In [ ]:
xgb_importance_comparison = (
    xgb_feature_importance
    .merge(
        xgb_mean_absolute_shap,
        on="feature",
        how="outer",
        validate="one_to_one"
    )
    .fillna(0)
)


xgb_importance_comparison[
    "gain_rank"
] = (
    xgb_importance_comparison[
        "importance"
    ]
    .rank(
        ascending=False,
        method="min"
    )
)


xgb_importance_comparison[
    "shap_rank"
] = (
    xgb_importance_comparison[
        "mean_absolute_shap"
    ]
    .rank(
        ascending=False,
        method="min"
    )
)


xgb_importance_comparison = (
    xgb_importance_comparison
    .sort_values(
        "shap_rank"
    )
    .reset_index(drop=True)
)


xgb_importance_comparison.to_csv(
    OUTPUT_DIR
    / "xgboost_importance_method_comparison.csv",
    index=False
)


xgb_importance_comparison.head(25)

Everything looks reliable now.

Higher external-source scores mean lower predicted default risk (both SHAP and gain importance).
Larger repayment obligations and debt burden tend to increase predicted default risk.
Shorter employment history generally increases predicted risk.
A greater fraction of late installment payments increases predicted default risk.
A higher proportion of scheduled payments actually paid reduces predicted risk. 
A higher proportion of previously refused applications increased predicted risk.
Male applicants tend to have higher predicted risk.
Being married tends to reduce predicted risk.
Higher education tends to reduce predicted risk.

The ranking for gains and SHAP are much more similar now. Gain-based importance and SHAP broadly agreed on the main sources of predictive information, although their precise rankings differed because gain measures split improvement while SHAP measures average applicant-level prediction impact.

In [ ]:
grouped_shap_records = []

for original_feature in pd.unique(
    processed_to_original_feature
):
    feature_positions = np.flatnonzero(
        processed_to_original_feature
        == original_feature
    )

    grouped_contribution = (
        xgb_shap_values[
            :,
            feature_positions
        ]
        .sum(axis=1)
    )

    grouped_shap_records.append({
        "feature": original_feature,

        "number_of_processed_columns": int(
            len(feature_positions)
        ),

        "mean_absolute_grouped_shap": float(
            np.abs(
                grouped_contribution
            ).mean()
        )
    })


xgb_grouped_shap_importance = (
    pd.DataFrame(
        grouped_shap_records
    )
    .sort_values(
        "mean_absolute_grouped_shap",
        ascending=False
    )
    .reset_index(drop=True)
)


xgb_grouped_shap_importance.to_csv(
    OUTPUT_DIR
    / "xgboost_grouped_shap_importance.csv",
    index=False
)


xgb_grouped_shap_importance.head(25)

In [ ]:
top_grouped_shap = (
    xgb_grouped_shap_importance
    .head(25)
    .sort_values(
        "mean_absolute_grouped_shap",
        ascending=True
    )
)


figure, axis = plt.subplots(
    figsize=(10, 9)
)

axis.barh(
    top_grouped_shap["feature"],
    top_grouped_shap[
        "mean_absolute_grouped_shap"
    ]
)

axis.set_xlabel(
    "Mean absolute grouped SHAP value"
)

axis.set_ylabel(
    "Original feature"
)

axis.set_title(
    "XGBoost ensemble component:\n"
    "Top original features by grouped SHAP importance"
)

axis.grid(
    axis="x",
    alpha=0.25
)

figure.tight_layout()

figure.savefig(
    OUTPUT_DIR
    / "xgboost_component_grouped_shap_importance.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close(figure)

In [ ]:
dependence_features = [
    "EXT_SOURCE_2",
    "ANNUITY_CREDIT_RATIO",
    "BUREAU_DEBT_CREDIT_RATIO_MAX",
    "INST_LATE_RATE"
]


for feature in dependence_features:
    feature_position = int(
        np.flatnonzero(
            xgb_clean_feature_names
            == feature
        )[0]
    )

    plt.figure(
        figsize=(8, 6)
    )

    shap.dependence_plot(
        ind=feature_position,
        shap_values=xgb_shap_values,
        features=xgb_shap_display_values,
        feature_names=xgb_clean_feature_names,
        interaction_index=None,
        show=False
    )

    plt.title(
        f"XGBoost component: SHAP dependence\n"
        f"{feature}"
    )

    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR
        / f"xgboost_shap_dependence_{feature}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close()

Grouped, there are four main sources of predictive information:
- external risk scores
- loan and debt burden
- historical repayment behavior
- applicant characteristics

In [ ]:
feature = "BUREAU_DEBT_CREDIT_RATIO_MAX"

feature_position = int(
    np.flatnonzero(
        xgb_clean_feature_names == feature
    )[0]
)

feature_values = (
    xgb_shap_display_values[
        :,
        feature_position
    ]
)

finite_values = feature_values[
    np.isfinite(feature_values)
]

upper_limit = np.quantile(
    finite_values,
    0.99
)

plt.figure(
    figsize=(8, 6)
)

plt.scatter(
    feature_values,
    xgb_shap_values[
        :,
        feature_position
    ],
    alpha=0.5,
    s=14
)

plt.xlim(
    finite_values.min(),
    upper_limit
)

plt.axhline(
    0,
    linestyle="--",
    linewidth=1
)

plt.xlabel(feature)

plt.ylabel(
    f"SHAP value for\n{feature}"
)

plt.title(
    "XGBoost component: SHAP dependence\n"
    f"{feature}, x-axis limited to 99th percentile"
)

plt.tight_layout()

plt.savefig(
    OUTPUT_DIR
    / "xgboost_shap_dependence_"
      "BUREAU_DEBT_CREDIT_RATIO_MAX_clipped.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()
plt.close()

Final model-comparison tables

The earlier logistic-regression and random-forest models used the V1
application-only pipeline and a single validation split.

The final boosting models used the V3 application-plus-history feature
pipelines and five-fold development OOF evaluation. These differences are
reported explicitly, so the table should be interpreted as project
progression rather than a perfectly controlled algorithm-only comparison.

In [ ]:
LEGACY_MODEL_RESULTS = {
    "Logistic regression": {
        "roc_auc": 0.629460950912641,

        # Average precision was not calculated in V1.
        "average_precision": np.nan,

        "feature_version": (
            "V1 application-only features; "
            "median/mode imputation and one-hot encoding"
        ),

        "evaluation_set": (
            "Single 80/20 validation split; "
            "logistic regression did not fully converge"
        )
    },

    "Random forest": {
        "roc_auc": 0.7107978692119733,

        # Average precision was not calculated in V1.
        "average_precision": np.nan,

        "feature_version": (
            "V1 application-only features; "
            "median/mode imputation and one-hot encoding"
        ),

        "evaluation_set": (
            "Single 80/20 validation split"
        )
    }
}

In [ ]:
legacy_results_check = pd.DataFrame([
    {
        "model": model_name,
        "roc_auc": result["roc_auc"],
        "average_precision": (
            result["average_precision"]
        ),
        "feature_version": (
            result["feature_version"]
        ),
        "evaluation_set": (
            result["evaluation_set"]
        )
    }

    for model_name, result
    in LEGACY_MODEL_RESULTS.items()
])

legacy_results_check

In [ ]:
V3_FINAL_MODEL_RESULTS = {
    "XGBoost": {
        "number_of_features": 160,
        "mean_cv_auc": 0.785054,
        "std_cv_auc": 0.002019,
        "minimum_fold_auc": 0.782012,

        "mean_cv_average_precision": 0.275723,
        "std_cv_average_precision": 0.007379,

        "mean_best_iteration": 1395.6,
        "mean_runtime_seconds": 70.932100,

        "oof_auc": 0.785042,
        "oof_average_precision": 0.275291
    },

    "LightGBM": {
        "number_of_features": 160,
        "mean_cv_auc": 0.784763,
        "std_cv_auc": 0.001897,
        "minimum_fold_auc": 0.781682,

        "mean_cv_average_precision": 0.274743,
        "std_cv_average_precision": 0.007165,

        "mean_best_iteration": 1739.6,
        "mean_runtime_seconds": 20.793971,

        "oof_auc": 0.784731,
        "oof_average_precision": 0.274246
    },

    "CatBoost": {
        "number_of_features": 157,
        "mean_cv_auc": 0.784349,
        "std_cv_auc": 0.001896,
        "minimum_fold_auc": 0.781936,

        "mean_cv_average_precision": 0.274929,
        "std_cv_average_precision": 0.008212,

        "mean_best_iteration": 2073.0,
        "mean_runtime_seconds": 120.818135,

        "oof_auc": 0.784321,
        "oof_average_precision": 0.274353
    }
}

In [ ]:
boosting_comparison_table = pd.DataFrame([
    {
        "model": model_name,
        **result
    }

    for model_name, result
    in V3_FINAL_MODEL_RESULTS.items()
])

In [ ]:
boosting_holdout_metrics = (
    holdout_performance
    .rename(
        columns={
            "candidate": "model"
        }
    )
    .query(
        "model != 'Equal-weight ensemble'"
    )
    .copy()
)

In [ ]:
boosting_comparison_table = (
    boosting_comparison_table
    .merge(
        boosting_holdout_metrics,
        on="model",
        how="left",
        validate="one_to_one"
    )
    .sort_values(
        "oof_auc",
        ascending=False
    )
    .reset_index(drop=True)
)

In [ ]:
boosting_comparison_table

boosting_comparison_table.to_csv(
    OUTPUT_DIR
    / "final_boosting_model_comparison.csv",
    index=False
)

In [ ]:
ensemble_comparison_row = pd.DataFrame([{
    "model": (
        "Equal-weight ensemble"
    ),

    "number_of_features": np.nan,

    "mean_cv_auc": np.nan,
    "std_cv_auc": np.nan,
    "minimum_fold_auc": np.nan,

    "mean_cv_average_precision": np.nan,
    "std_cv_average_precision": np.nan,

    "mean_best_iteration": np.nan,
    "mean_runtime_seconds": np.nan,

    "oof_auc": (
        V3_SELECTION_RESULTS[
            "Final_ensemble_oof_auc"
        ]
    ),

    "oof_average_precision": (
        V3_SELECTION_RESULTS[
            "Final_ensemble_oof_ap"
        ]
    ),

    "holdout_roc_auc": (
        final_holdout_auc
    ),

    "holdout_average_precision": (
        final_holdout_ap
    )
}])

In [ ]:
final_boosting_and_ensemble_table = (
    pd.concat(
        [
            boosting_comparison_table,
            ensemble_comparison_row
        ],
        ignore_index=True
    )
    .sort_values(
        "oof_auc",
        ascending=False
    )
    .reset_index(drop=True)
)

final_boosting_and_ensemble_table

In [ ]:
final_boosting_and_ensemble_table.to_csv(
    OUTPUT_DIR
    / "final_boosting_and_ensemble_comparison.csv",
    index=False
)

In [ ]:
boosting_presentation_table = (
    final_boosting_and_ensemble_table[
        [
            "model",
            "number_of_features",
            "oof_auc",
            "oof_average_precision",
            "holdout_roc_auc",
            "holdout_average_precision",
            "mean_runtime_seconds"
        ]
    ]
    .copy()
)

boosting_presentation_table

In [ ]:
boosting_presentation_display = (
    boosting_presentation_table.copy()
)

numeric_columns_to_format = [
    "oof_auc",
    "oof_average_precision",
    "holdout_roc_auc",
    "holdout_average_precision"
]

for column in numeric_columns_to_format:
    boosting_presentation_display[
        column
    ] = (
        boosting_presentation_display[
            column
        ]
        .map(
            lambda value:
            f"{value:.6f}"
            if pd.notna(value)
            else "—"
        )
    )

boosting_presentation_display[
    "mean_runtime_seconds"
] = (
    boosting_presentation_display[
        "mean_runtime_seconds"
    ]
    .map(
        lambda value:
        f"{value:.1f}"
        if pd.notna(value)
        else "—"
    )
)

boosting_presentation_display

In [ ]:
legacy_project_rows = pd.DataFrame([
    {
        "model": model_name,

        "model_family": (
            "Baseline / earlier model"
        ),

        "feature_version": (
            result[
                "feature_version"
            ]
        ),

        "evaluation_basis": (
            result[
                "evaluation_set"
            ]
        ),

        "roc_auc": (
            result[
                "roc_auc"
            ]
        ),

        "average_precision": (
            result[
                "average_precision"
            ]
        ),

        "holdout_roc_auc": np.nan,
        "holdout_average_precision": np.nan
    }

    for model_name, result
    in LEGACY_MODEL_RESULTS.items()
])

In [ ]:
final_project_rows = pd.DataFrame([
    {
        "model": "XGBoost",

        "model_family": (
            "Final gradient boosting"
        ),

        "feature_version": (
            "V3 final application + bureau + "
            "previous applications + installments"
        ),

        "evaluation_basis": (
            "Five-fold development OOF"
        ),

        "roc_auc": 0.785042,
        "average_precision": 0.275291,

        "holdout_roc_auc": 0.787571,
        "holdout_average_precision": 0.287086
    },

    {
        "model": "LightGBM",

        "model_family": (
            "Final gradient boosting"
        ),

        "feature_version": (
            "V3 final application + bureau + "
            "previous applications + installments"
        ),

        "evaluation_basis": (
            "Five-fold development OOF"
        ),

        "roc_auc": 0.784731,
        "average_precision": 0.274246,

        "holdout_roc_auc": 0.787839,
        "holdout_average_precision": 0.287102
    },

    {
        "model": "CatBoost",

        "model_family": (
            "Final gradient boosting"
        ),

        "feature_version": (
            "V3 final application + bureau + "
            "previous applications + installments"
        ),

        "evaluation_basis": (
            "Five-fold development OOF"
        ),

        "roc_auc": 0.784321,
        "average_precision": 0.274353,

        "holdout_roc_auc": 0.787611,
        "holdout_average_precision": 0.288693
    },

    {
        "model": (
            "Equal-weight ensemble"
        ),

        "model_family": (
            "Final ensemble"
        ),

        "feature_version": (
            "V3 final model-specific application "
            "+ historical feature pipelines"
        ),

        "evaluation_basis": (
            "Five-fold development OOF"
        ),

        "roc_auc": 0.786843,
        "average_precision": 0.278130,

        "holdout_roc_auc": 0.789454,
        "holdout_average_precision": 0.291160
    }
])

In [ ]:
project_model_comparison = (
    pd.concat(
        [
            legacy_project_rows,
            final_project_rows
        ],
        ignore_index=True
    )
    .sort_values(
        "roc_auc",
        ascending=False,
        na_position="last"
    )
    .reset_index(drop=True)
)

In [ ]:
pd.set_option(
    "display.max_colwidth",
    None
)

project_model_comparison

In [ ]:
project_model_comparison.to_csv(
    OUTPUT_DIR
    / "complete_project_model_comparison.csv",
    index=False
)

In [ ]:
project_progression_table = pd.DataFrame({
    "stage": [
        "V1 baseline",
        "V1 tree model",
        "V1 initial boosting",
        "V3 final individual model",
        "V3 final ensemble",
        "V4 untouched holdout"
    ],

    "model": [
        "Logistic regression",
        "Random forest",
        "XGBoost",
        "XGBoost",
        "Equal-weight ensemble",
        "Equal-weight ensemble"
    ],

    "evaluation": [
        "Single validation split",
        "Single validation split",
        "Single validation split",
        "Five-fold development OOF",
        "Five-fold development OOF",
        "Untouched holdout"
    ],

    "roc_auc": [
        0.629460950912641,
        0.7107978692119733,
        0.7585567489174014,
        0.785042,
        0.786843,
        0.789454
    ],

    "average_precision": [
        np.nan,
        np.nan,
        np.nan,
        0.275291,
        0.278130,
        0.291160
    ]
})

In [ ]:
project_progression_table.to_csv(
    OUTPUT_DIR
    / "project_model_progression.csv",
    index=False
)

project_progression_table

The final operating threshold was selected by maximizing F2 on development
OOF ensemble probabilities. This gives greater emphasis to identifying
actual defaults while continuing to account for precision.

The threshold was then applied once to the untouched holdout predictions.

In [ ]:
final_threshold_summary = pd.DataFrame({
    "item": [
        "Selection dataset",
        "Selection rule",
        "Locked threshold",

        "Development OOF predicted-default rate",
        "Development OOF precision",
        "Development OOF recall",
        "Development OOF specificity",
        "Development OOF F1",
        "Development OOF F2",
        "Development OOF balanced accuracy",

        "Holdout predicted-default rate",
        "Holdout precision",
        "Holdout recall",
        "Holdout specificity",
        "Holdout F1",
        "Holdout F2",
        "Holdout balanced accuracy"
    ],

    "value": [
        "Development OOF predictions",

        THRESHOLD_SELECTION_RULE,

        LOCKED_CLASSIFICATION_THRESHOLD,

        locked_development_threshold_result[
            "predicted_default_rate"
        ],

        locked_development_threshold_result[
            "precision"
        ],

        locked_development_threshold_result[
            "recall"
        ],

        locked_development_threshold_result[
            "specificity"
        ],

        locked_development_threshold_result[
            "f1"
        ],

        locked_development_threshold_result[
            "f2"
        ],

        locked_development_threshold_result[
            "balanced_accuracy"
        ],

        locked_holdout_threshold_result[
            "predicted_default_rate"
        ],

        locked_holdout_threshold_result[
            "precision"
        ],

        locked_holdout_threshold_result[
            "recall"
        ],

        locked_holdout_threshold_result[
            "specificity"
        ],

        locked_holdout_threshold_result[
            "f1"
        ],

        locked_holdout_threshold_result[
            "f2"
        ],

        locked_holdout_threshold_result[
            "balanced_accuracy"
        ]
    ]
})

In [ ]:
final_threshold_summary.to_csv(
    OUTPUT_DIR
    / "final_threshold_summary.csv",
    index=False
)

final_threshold_summary

In [ ]:
threshold_performance_comparison = pd.DataFrame({
    "dataset": [
        "Development OOF",
        "Untouched holdout"
    ],

    "threshold": [
        LOCKED_CLASSIFICATION_THRESHOLD,
        LOCKED_CLASSIFICATION_THRESHOLD
    ],

    "predicted_default_rate": [
        locked_development_threshold_result[
            "predicted_default_rate"
        ],

        locked_holdout_threshold_result[
            "predicted_default_rate"
        ]
    ],

    "precision": [
        locked_development_threshold_result[
            "precision"
        ],

        locked_holdout_threshold_result[
            "precision"
        ]
    ],

    "recall": [
        locked_development_threshold_result[
            "recall"
        ],

        locked_holdout_threshold_result[
            "recall"
        ]
    ],

    "specificity": [
        locked_development_threshold_result[
            "specificity"
        ],

        locked_holdout_threshold_result[
            "specificity"
        ]
    ],

    "f1": [
        locked_development_threshold_result[
            "f1"
        ],

        locked_holdout_threshold_result[
            "f1"
        ]
    ],

    "f2": [
        locked_development_threshold_result[
            "f2"
        ],

        locked_holdout_threshold_result[
            "f2"
        ]
    ],

    "balanced_accuracy": [
        locked_development_threshold_result[
            "balanced_accuracy"
        ],

        locked_holdout_threshold_result[
            "balanced_accuracy"
        ]
    ]
})

In [ ]:
threshold_performance_comparison.to_csv(
    OUTPUT_DIR
    / "threshold_performance_development_vs_holdout.csv",
    index=False
)

threshold_performance_comparison

In [ ]:
final_model_selection_record = pd.DataFrame({
    "decision": [
        "Final probability model",
        "Ensemble members",
        "Ensemble weights",
        "Development OOF ROC AUC",
        "Development OOF average precision",
        "Untouched holdout ROC AUC",
        "Untouched holdout average precision",
        "Classification-threshold rule",
        "Locked classification threshold"
    ],

    "result": [
        "Equal-weight gradient boosting ensemble",

        (
            "XGBoost, LightGBM, CatBoost"
        ),

        (
            "One-third per model"
        ),

        (
            V3_SELECTION_RESULTS[
                "Final_ensemble_oof_auc"
            ]
        ),

        (
            V3_SELECTION_RESULTS[
                "Final_ensemble_oof_ap"
            ]
        ),

        final_holdout_auc,
        final_holdout_ap,

        THRESHOLD_SELECTION_RULE,

        LOCKED_CLASSIFICATION_THRESHOLD
    ]
})

In [ ]:
final_model_selection_record.to_csv(
    OUTPUT_DIR
    / "final_model_selection_record.csv",
    index=False
)

final_model_selection_record

The original application-only logistic-regression baseline achieved a
validation ROC AUC of approximately 0.6295, while the original random forest
achieved approximately 0.7108.

The final XGBoost, LightGBM, and CatBoost pipelines each achieved development
OOF ROC AUC values near 0.785. Their equal-weight probability ensemble
improved development OOF performance to 0.7868 and achieved a final untouched
holdout ROC AUC of 0.7895 with average precision of 0.2912.

The operating threshold was selected independently by maximizing F2 on
development OOF predictions. Its nearly identical development and holdout
precision, recall, specificity, and F2 provide additional evidence that the
final pipeline generalized reliably.

Although the three gradient-boosting models achieved similar individual performance, their equal-weight probability ensemble consistently outperformed each component on both development OOF predictions and the untouched holdout set. This suggests that the models captured partially complementary prediction errors despite their similar overall discrimination.

LightGBM is about 3.4 times faster than XGBoost and about 5.8 times faster than CatBoost while producing almost identical AUC.

Progression:
V1 logistic regression AUC: 0.6295
V1 random forest AUC:       0.7108
V1 initial XGBoost AUC:     0.7586
V3 final XGBoost OOF AUC:   0.7850
V3 ensemble OOF AUC:        0.7868
V4 ensemble holdout AUC:    0.7895

The OOF-selected threshold of 0.0878 produced nearly identical operating characteristics on development and holdout data. It identified approximately 68% of defaults while retaining approximately 75% specificity and flagging about 29% of applicants. The close agreement confirms that the selected classification threshold was stable rather than overfit to the development predictions.

Final probability model:
Equal-weight XGBoost, LightGBM, and CatBoost ensemble

Weights:
One-third per model

Development OOF:
AUC = 0.786843
AP  = 0.278130

Untouched holdout:
AUC = 0.789454
AP  = 0.291160

Classification threshold:
0.087754

Selection method:
Maximum F2 on development OOF predictions

Refit the locked pipelines on all labeled data

The final evaluation has now been completed and recorded. The development
and holdout observations can therefore be recombined for the final production
fit.

The same locked feature sets, hyperparameters, fixed boosting iterations,
and equal ensemble weights are used. No additional feature selection,
hyperparameter tuning, early stopping, or threshold optimization is
performed.

The classification threshold is not used during model fitting. It is also
not applied to the Kaggle predictions, which must remain continuous
probabilities.

In [ ]:
X_full_xgb_final = (
    pd.concat(
        [
            X_dev_xgb_final,
            X_hold_xgb_final
        ],
        axis=0
    )
    .sort_index()
)


X_full_catboost_final = (
    pd.concat(
        [
            X_dev_catboost_final,
            X_hold_catboost_final
        ],
        axis=0
    )
    .sort_index()
)


X_full_lightgbm_final = (
    pd.concat(
        [
            X_dev_lightgbm_final,
            X_hold_lightgbm_final
        ],
        axis=0
    )
    .sort_index()
)


y_full = (
    pd.concat(
        [
            y_dev,
            y_hold
        ],
        axis=0
    )
    .sort_index()
)

In [ ]:
assert len(y_full) == len(data)

assert X_full_xgb_final.shape == (
    len(data),
    160
)

assert X_full_catboost_final.shape == (
    len(data),
    157
)

assert X_full_lightgbm_final.shape == (
    len(data),
    160
)

In [ ]:
assert X_full_xgb_final.index.equals(
    data.index
)

assert X_full_catboost_final.index.equals(
    data.index
)

assert X_full_lightgbm_final.index.equals(
    data.index
)

assert y_full.index.equals(
    data.index
)

In [ ]:
assert list(
    X_full_xgb_final.columns
) == list(
    data_test_xgb_final.columns
)

assert list(
    X_full_catboost_final.columns
) == list(
    data_test_catboost_final.columns
)

assert list(
    X_full_lightgbm_final.columns
) == list(
    data_test_lightgbm_final.columns
)

In [ ]:
full_data_summary = pd.DataFrame({
    "pipeline": [
        "XGBoost",
        "CatBoost",
        "LightGBM"
    ],

    "training_rows": [
        X_full_xgb_final.shape[0],
        X_full_catboost_final.shape[0],
        X_full_lightgbm_final.shape[0]
    ],

    "training_features": [
        X_full_xgb_final.shape[1],
        X_full_catboost_final.shape[1],
        X_full_lightgbm_final.shape[1]
    ],

    "test_rows": [
        data_test_xgb_final.shape[0],
        data_test_catboost_final.shape[0],
        data_test_lightgbm_final.shape[0]
    ]
})

full_data_summary

In [ ]:
print(
    "Full labeled observations:",
    f"{len(y_full):,}"
)

print(
    "Full-data defaults:",
    f"{int(y_full.sum()):,}"
)

print(
    "Full-data default rate:",
    f"{y_full.mean():.4%}"
)

In [ ]:
#memory cleanup
objects_to_remove = [
    "xgb_training_matrix",
    "xgb_holdout_matrix",

    "X_dev_catboost_prepared",
    "X_hold_catboost_prepared",

    "X_dev_lightgbm_prepared",
    "X_hold_lightgbm_prepared",

    "xgb_shap_matrix",
    "xgb_shap_matrix_dense",
    "xgb_shap_display_values",
    "xgb_shap_values",
    "xgb_native_contributions",
    "xgb_shap_dmatrix"
]


for object_name in objects_to_remove:
    if object_name in globals():
        del globals()[
            object_name
        ]


collected_objects = gc.collect()

print(
    "Garbage-collection result:",
    collected_objects
)

In [ ]:
xgb_full_preprocessor = (
    build_xgb_preprocessor(
        selected_features=(
            X_full_xgb_final.columns
        ),
        categorical_features=(
            categorical_features
        )
    )
)

In [ ]:
xgb_full_training_matrix = (
    xgb_full_preprocessor
    .fit_transform(
        X_full_xgb_final
    )
)


xgb_test_matrix = (
    xgb_full_preprocessor
    .transform(
        data_test_xgb_final
    )
)

In [ ]:
print(
    "Full XGBoost training matrix:",
    xgb_full_training_matrix.shape
)

print(
    "XGBoost Kaggle test matrix:",
    xgb_test_matrix.shape
)


assert xgb_full_training_matrix.shape[0] == (
    len(y_full)
)

assert xgb_test_matrix.shape[0] == (
    len(kaggle_test)
)

assert xgb_full_training_matrix.shape[1] == (
    xgb_test_matrix.shape[1]
)

In [ ]:
xgb_full_model = XGBClassifier(
    **XGB_FINAL_SETTINGS
)


xgb_full_start_time = (
    time.perf_counter()
)


xgb_full_model.fit(
    xgb_full_training_matrix,
    y_full,
    verbose=False
)


xgb_full_training_seconds = (
    time.perf_counter()
    - xgb_full_start_time
)

In [ ]:
xgb_test_probability = (
    xgb_full_model.predict_proba(
        xgb_test_matrix
    )[:, 1]
)

In [ ]:
print(
    "Full-data XGBoost training time:",
    f"{xgb_full_training_seconds:.1f} seconds"
)

print(
    "XGBoost test mean probability:",
    f"{xgb_test_probability.mean():.6f}"
)

In [ ]:
(
    X_full_catboost_prepared,
    data_test_catboost_prepared,
    catboost_full_categorical_features
) = prepare_catboost_pair(
    training_frame=(
        X_full_catboost_final
    ),
    prediction_frame=(
        data_test_catboost_final
    ),
    categorical_features=(
        categorical_features
    )
)

In [ ]:
assert X_full_catboost_prepared.shape == (
    len(y_full),
    157
)

assert data_test_catboost_prepared.shape == (
    len(kaggle_test),
    157
)

assert list(
    X_full_catboost_prepared.columns
) == list(
    data_test_catboost_prepared.columns
)


print(
    "CatBoost categorical features:",
    len(
        catboost_full_categorical_features
    )
)

In [ ]:
catboost_full_model = (
    CatBoostClassifier(
        **CATBOOST_FINAL_SETTINGS
    )
)


catboost_full_start_time = (
    time.perf_counter()
)


catboost_full_model.fit(
    X_full_catboost_prepared,
    y_full,
    cat_features=(
        catboost_full_categorical_features
    ),
    verbose=False
)


catboost_full_training_seconds = (
    time.perf_counter()
    - catboost_full_start_time
)

In [ ]:
catboost_test_probability = (
    catboost_full_model.predict_proba(
        data_test_catboost_prepared
    )[:, 1]
)

In [ ]:
print(
    "Full-data CatBoost training time:",
    f"{catboost_full_training_seconds:.1f} seconds"
)

print(
    "CatBoost test mean probability:",
    f"{catboost_test_probability.mean():.6f}"
)

In [ ]:
(
    X_full_lightgbm_prepared,
    data_test_lightgbm_prepared,
    lightgbm_full_categorical_features
) = prepare_lightgbm_pair(
    training_frame=(
        X_full_lightgbm_final
    ),
    prediction_frame=(
        data_test_lightgbm_final
    ),
    categorical_features=(
        categorical_features
    )
)

In [ ]:
assert X_full_lightgbm_prepared.shape == (
    len(y_full),
    160
)

assert data_test_lightgbm_prepared.shape == (
    len(kaggle_test),
    160
)

assert list(
    X_full_lightgbm_prepared.columns
) == list(
    data_test_lightgbm_prepared.columns
)


print(
    "LightGBM categorical features:",
    len(
        lightgbm_full_categorical_features
    )
)

In [ ]:
for feature in (
    lightgbm_full_categorical_features
):
    assert isinstance(
        X_full_lightgbm_prepared[
            feature
        ].dtype,
        pd.CategoricalDtype
    )

    assert isinstance(
        data_test_lightgbm_prepared[
            feature
        ].dtype,
        pd.CategoricalDtype
    )

In [ ]:
lightgbm_full_model = (
    LGBMClassifier(
        **LIGHTGBM_FINAL_SETTINGS
    )
)


lightgbm_full_start_time = (
    time.perf_counter()
)


lightgbm_full_model.fit(
    X_full_lightgbm_prepared,
    y_full,
    categorical_feature=(
        lightgbm_full_categorical_features
    )
)


lightgbm_full_training_seconds = (
    time.perf_counter()
    - lightgbm_full_start_time
)

In [ ]:
lightgbm_test_probability = (
    lightgbm_full_model.predict_proba(
        data_test_lightgbm_prepared
    )[:, 1]
)

In [ ]:
print(
    "Full-data LightGBM training time:",
    f"{lightgbm_full_training_seconds:.1f} seconds"
)

print(
    "LightGBM test mean probability:",
    f"{lightgbm_test_probability.mean():.6f}"
)

In [ ]:
final_test_probability = (
    final_ensemble_weights[
        "XGBoost"
    ]
    * xgb_test_probability

    + final_ensemble_weights[
        "LightGBM"
    ]
    * lightgbm_test_probability

    + final_ensemble_weights[
        "CatBoost"
    ]
    * catboost_test_probability
)

In [ ]:
test_probability_dictionary = {
    "XGBoost": (
        xgb_test_probability
    ),

    "LightGBM": (
        lightgbm_test_probability
    ),

    "CatBoost": (
        catboost_test_probability
    ),

    "Equal-weight ensemble": (
        final_test_probability
    )
}

In [ ]:
test_probability_summary_records = []


for model_name, probability in (
    test_probability_dictionary.items()
):
    probability = np.asarray(
        probability
    )

    assert len(probability) == len(
        kaggle_test
    )

    assert np.isfinite(
        probability
    ).all()

    assert (
        probability >= 0
    ).all()

    assert (
        probability <= 1
    ).all()


    test_probability_summary_records.append({
        "model": model_name,

        "observations": int(
            len(probability)
        ),

        "minimum_probability": float(
            probability.min()
        ),

        "mean_probability": float(
            probability.mean()
        ),

        "median_probability": float(
            np.median(
                probability
            )
        ),

        "maximum_probability": float(
            probability.max()
        ),

        "standard_deviation": float(
            probability.std()
        )
    })

In [ ]:
test_probability_summary = pd.DataFrame(
    test_probability_summary_records
)


test_probability_summary.to_csv(
    OUTPUT_DIR
    / "full_data_test_probability_summary.csv",
    index=False
)


test_probability_summary

In [ ]:
test_prediction_correlation = pd.DataFrame({
    "XGBoost": (
        xgb_test_probability
    ),

    "LightGBM": (
        lightgbm_test_probability
    ),

    "CatBoost": (
        catboost_test_probability
    )
}).corr()


test_prediction_correlation.to_csv(
    OUTPUT_DIR
    / "full_data_test_prediction_correlations.csv"
)


test_prediction_correlation

In [ ]:
full_data_runtime = pd.DataFrame({
    "model": [
        "XGBoost",
        "LightGBM",
        "CatBoost"
    ],

    "training_rows": [
        len(y_full),
        len(y_full),
        len(y_full)
    ],

    "number_of_features": [
        X_full_xgb_final.shape[1],
        X_full_lightgbm_final.shape[1],
        X_full_catboost_final.shape[1]
    ],

    "fixed_iterations": [
        FINAL_ITERATIONS[
            "XGBoost"
        ],

        FINAL_ITERATIONS[
            "LightGBM"
        ],

        FINAL_ITERATIONS[
            "CatBoost"
        ]
    ],

    "training_seconds": [
        xgb_full_training_seconds,
        lightgbm_full_training_seconds,
        catboost_full_training_seconds
    ]
})

In [ ]:
full_data_runtime[
    "training_minutes"
] = (
    full_data_runtime[
        "training_seconds"
    ]
    / 60
)


full_data_runtime.to_csv(
    OUTPUT_DIR
    / "full_data_training_runtime.csv",
    index=False
)


full_data_runtime

In [ ]:
final_refit_summary = pd.DataFrame({
    "item": [
        "Full labeled observations",
        "Full labeled defaults",
        "Full labeled default rate",

        "XGBoost raw features",
        "CatBoost raw features",
        "LightGBM raw features",

        "XGBoost fixed iterations",
        "CatBoost fixed iterations",
        "LightGBM fixed iterations",

        "Ensemble members",
        "Ensemble weighting",

        "Kaggle test observations"
    ],

    "value": [
        len(y_full),
        int(y_full.sum()),
        float(y_full.mean()),

        X_full_xgb_final.shape[1],
        X_full_catboost_final.shape[1],
        X_full_lightgbm_final.shape[1],

        FINAL_ITERATIONS[
            "XGBoost"
        ],

        FINAL_ITERATIONS[
            "CatBoost"
        ],

        FINAL_ITERATIONS[
            "LightGBM"
        ],

        (
            "XGBoost, LightGBM, CatBoost"
        ),

        (
            "Equal weight: one-third each"
        ),

        len(kaggle_test)
    ]
})

In [ ]:
final_refit_summary.to_csv(
    OUTPUT_DIR
    / "final_full_data_refit_summary.csv",
    index=False
)


final_refit_summary

In [ ]:
assert "early_stopping_rounds" not in (
    XGB_FINAL_SETTINGS
)

assert XGB_FINAL_SETTINGS[
    "n_estimators"
] == 1396

assert CATBOOST_FINAL_SETTINGS[
    "iterations"
] == 2073

assert LIGHTGBM_FINAL_SETTINGS[
    "n_estimators"
] == 1740

In [ ]:
assert len(
    final_test_probability
) == 48744

assert np.isfinite(
    final_test_probability
).all()

assert final_test_probability.min() >= 0
assert final_test_probability.max() <= 1

In [ ]:
print(
    "=" * 60
)

print(
    "FINAL FULL-DATA REFIT COMPLETE"
)

print(
    "=" * 60
)

print(
    "Training observations:",
    f"{len(y_full):,}"
)

print(
    "Kaggle test observations:",
    f"{len(final_test_probability):,}"
)

print(
    "XGBoost iterations:",
    FINAL_ITERATIONS["XGBoost"]
)

print(
    "LightGBM iterations:",
    FINAL_ITERATIONS["LightGBM"]
)

print(
    "CatBoost iterations:",
    FINAL_ITERATIONS["CatBoost"]
)

print()

print(
    "Final ensemble probability range:",
    (
        f"{final_test_probability.min():.6f}"
        " to "
        f"{final_test_probability.max():.6f}"
    )
)

print(
    "Final ensemble mean probability:",
    f"{final_test_probability.mean():.6f}"
)

print(
    "=" * 60
)

Full-data refit conclusion

The locked XGBoost, LightGBM, and CatBoost pipelines were refitted on all
307,511 labeled applicants using the iteration counts selected during V3
cross-validation.

The three models produced continuous default probabilities for all 48,744
Kaggle test applicants. Their equal-weight average is the final submission
prediction.

The maximum-F2 classification threshold is not applied to these predictions
because Kaggle evaluates continuous probability rankings rather than hard
default classifications.

(In the final full-data refit, XGBoost had the shortest measured model-fitting time at 19.7 seconds, followed by LightGBM at 27.9 seconds and CatBoost at 207.0 seconds. Runtime comparisons should be interpreted cautiously because preprocessing and validation overhead differed across model pipelines and notebook stages.)

Super consistent prediction results are great!

Really happy with how robust the models are.
I know overfitting can be a challenging for gradient boosting, but we're using limited complexity (XGBoost has max_depth = 3, CatBoost has depth = 4, LightGBM has num_leaves = 15) and low learning rates (XGBoost: 0.05, CatBoost: 0.05, LightGBM: 0.02). There are also min-obs. restrictions, column/row subsampling, and regularization. I guess the large dataset also helps a LOT. The feature development process was also conservative. 

The pipeline showed strong internal generalization and stability across cross-validation folds and an untouched random holdout drawn from the same source population.

Create the final Kaggle submission

The final submission uses the equal-weight average of the full-data XGBoost,
LightGBM, and CatBoost probability predictions.

The locked classification threshold is not applied. The Home Credit
competition evaluates continuous default-risk probabilities using ROC AUC,
so each test applicant receives a probability in the `TARGET` column.

In [ ]:
SUBMISSION_DIR = (
    OUTPUT_DIR
    / "submissions"
)

SUBMISSION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

SUBMISSION_DIR

In [ ]:
final_submission = pd.DataFrame({
    "SK_ID_CURR": (
        kaggle_test[
            "SK_ID_CURR"
        ]
        .astype("int32")
        .to_numpy()
    ),

    "TARGET": (
        np.asarray(
            final_test_probability,
            dtype="float64"
        )
    )
})

In [ ]:
final_submission.head()

In [ ]:
assert final_submission.shape == (
    len(kaggle_test),
    2
)

assert list(
    final_submission.columns
) == [
    "SK_ID_CURR",
    "TARGET"
]

In [ ]:
assert np.array_equal(
    final_submission[
        "SK_ID_CURR"
    ].to_numpy(),

    kaggle_test[
        "SK_ID_CURR"
    ].astype("int32").to_numpy()
)

In [ ]:
assert final_submission[
    "SK_ID_CURR"
].notna().all()

assert final_submission[
    "SK_ID_CURR"
].is_unique

assert final_submission[
    "SK_ID_CURR"
].nunique() == len(
    kaggle_test
)

In [ ]:
assert final_submission[
    "TARGET"
].notna().all()

assert np.isfinite(
    final_submission[
        "TARGET"
    ]
).all()

assert final_submission[
    "TARGET"
].between(
    0,
    1,
    inclusive="both"
).all()

In [ ]:
assert final_submission[
    "TARGET"
].nunique() > 100

In [ ]:
assert len(
    final_submission
) == 48_744

In [ ]:
submission_probability_summary = (
    final_submission[
        "TARGET"
    ]
    .describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)

submission_probability_summary

In [ ]:
assert np.allclose(
    final_submission[
        "TARGET"
    ].to_numpy(),

    final_test_probability,
    atol=0,
    rtol=0
)

In [ ]:
FINAL_SUBMISSION_PATH = (
    SUBMISSION_DIR
    / "home_credit_v4_final_ensemble_submission.csv"
)

In [ ]:
final_submission.to_csv(
    FINAL_SUBMISSION_PATH,
    index=False,
    float_format="%.10f"
)

In [ ]:
print(
    "Final submission saved to:"
)

print(
    FINAL_SUBMISSION_PATH.resolve()
)

In [ ]:
saved_submission_check = pd.read_csv(
    FINAL_SUBMISSION_PATH
)

In [ ]:
assert saved_submission_check.shape == (
    48_744,
    2
)

assert list(
    saved_submission_check.columns
) == [
    "SK_ID_CURR",
    "TARGET"
]

assert saved_submission_check[
    "SK_ID_CURR"
].is_unique

assert saved_submission_check[
    "TARGET"
].between(
    0,
    1,
    inclusive="both"
).all()

In [ ]:
assert np.allclose(
    saved_submission_check[
        "TARGET"
    ].to_numpy(),

    final_test_probability,
    atol=5e-11,
    rtol=0
)

In [ ]:
saved_submission_check.head()
saved_submission_check.tail()

In [ ]:
submission_metadata = pd.DataFrame({
    "item": [
        "Submission filename",
        "Prediction model",
        "Component models",
        "Ensemble weights",
        "Training observations",
        "Test observations",
        "Minimum prediction",
        "Mean prediction",
        "Median prediction",
        "Maximum prediction",
        "Standard deviation",
        "Threshold applied",
        "Submission columns"
    ],

    "value": [
        FINAL_SUBMISSION_PATH.name,

        (
            "Equal-weight gradient "
            "boosting ensemble"
        ),

        (
            "XGBoost, LightGBM, "
            "CatBoost"
        ),

        (
            "One-third per model"
        ),

        len(y_full),

        len(final_submission),

        float(
            final_submission[
                "TARGET"
            ].min()
        ),

        float(
            final_submission[
                "TARGET"
            ].mean()
        ),

        float(
            final_submission[
                "TARGET"
            ].median()
        ),

        float(
            final_submission[
                "TARGET"
            ].max()
        ),

        float(
            final_submission[
                "TARGET"
            ].std()
        ),

        "No",

        "SK_ID_CURR, TARGET"
    ]
})

In [ ]:
submission_metadata.to_csv(
    OUTPUT_DIR
    / "final_submission_metadata.csv",
    index=False
)

submission_metadata

In [ ]:
test_prediction_components = pd.DataFrame({
    "SK_ID_CURR": (
        kaggle_test[
            "SK_ID_CURR"
        ]
        .astype("int32")
        .to_numpy()
    ),

    "XGBOOST_PROBABILITY": (
        xgb_test_probability
    ),

    "LIGHTGBM_PROBABILITY": (
        lightgbm_test_probability
    ),

    "CATBOOST_PROBABILITY": (
        catboost_test_probability
    ),

    "ENSEMBLE_PROBABILITY": (
        final_test_probability
    )
})

In [ ]:
test_prediction_components.to_csv(
    OUTPUT_DIR
    / "final_test_prediction_components.csv",
    index=False,
    float_format="%.10f"
)

In [ ]:
assert np.allclose(
    test_prediction_components[
        "ENSEMBLE_PROBABILITY"
    ],

    final_submission[
        "TARGET"
    ]
)

In [ ]:
submission_file_size_bytes = (
    FINAL_SUBMISSION_PATH
    .stat()
    .st_size
)

In [ ]:
print(
    "=" * 65
)

print(
    "FINAL KAGGLE SUBMISSION COMPLETE"
)

print(
    "=" * 65
)

print(
    "File:",
    FINAL_SUBMISSION_PATH.name
)

print(
    "Rows:",
    f"{len(final_submission):,}"
)

print(
    "Columns:",
    list(
        final_submission.columns
    )
)

print(
    "Unique applicant IDs:",
    f"{final_submission['SK_ID_CURR'].nunique():,}"
)

print(
    "Probability range:",
    (
        f"{final_submission['TARGET'].min():.6f}"
        " to "
        f"{final_submission['TARGET'].max():.6f}"
    )
)

print(
    "Mean probability:",
    f"{final_submission['TARGET'].mean():.6f}"
)

print(
    "File size:",
    f"{submission_file_size_bytes / 1024:.1f} KB"
)

print(
    "Threshold applied:",
    "No"
)

print(
    "=" * 65
)

The final Kaggle submission contains one continuous ensemble default-risk
probability for each of the 48,744 applicants in `application_test.csv`.

The predictions were generated by refitting the locked XGBoost, LightGBM,
and CatBoost pipelines on all labeled observations and averaging their
probabilities with equal weights. No classification threshold was applied.

The file `home_credit_v4_final_ensemble_submission.csv` is the selected
submission file for Kaggle.

In [ ]:
import shutil
from pathlib import Path


RESULTS_DIR = (
    OUTPUT_DIR
    / "results"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print(
    "Final results directory:"
)

print(
    RESULTS_DIR.resolve()
)

In [ ]:
train_missing_count = (
    data
    .isna()
    .sum()
)

test_missing_count = (
    kaggle_test
    .isna()
    .sum()
)


all_application_columns = sorted(
    set(data.columns)
    | set(kaggle_test.columns)
)


missingness_summary = pd.DataFrame({
    "feature": (
        all_application_columns
    )
})

In [ ]:
missingness_summary[
    "train_missing_count"
] = (
    missingness_summary[
        "feature"
    ]
    .map(
        train_missing_count
    )
    .fillna(0)
    .astype(int)
)


missingness_summary[
    "train_missing_rate"
] = (
    missingness_summary[
        "train_missing_count"
    ]
    / len(data)
)


missingness_summary[
    "test_missing_count"
] = (
    missingness_summary[
        "feature"
    ]
    .map(
        test_missing_count
    )
    .fillna(0)
    .astype(int)
)


missingness_summary[
    "test_missing_rate"
] = (
    missingness_summary[
        "test_missing_count"
    ]
    / len(kaggle_test)
)

In [ ]:
missingness_summary = (
    missingness_summary
    .sort_values(
        [
            "train_missing_rate",
            "test_missing_rate"
        ],
        ascending=False
    )
    .reset_index(drop=True)
)

In [ ]:
MISSINGNESS_RESULTS_PATH = (
    RESULTS_DIR
    / "01_missingness_summary.csv"
)


missingness_summary.to_csv(
    MISSINGNESS_RESULTS_PATH,
    index=False
)


missingness_summary.head(25)

In [ ]:
target_distribution_table = (
    y_full
    .value_counts()
    .sort_index()
    .rename_axis("target")
    .reset_index(name="count")
)


target_distribution_table[
    "label"
] = (
    target_distribution_table[
        "target"
    ]
    .map({
        0: "Non-default",
        1: "Default"
    })
)


target_distribution_table[
    "percentage"
] = (
    target_distribution_table[
        "count"
    ]
    / len(y_full)
    * 100
)

In [ ]:
figure, axis = plt.subplots(
    figsize=(8, 6)
)


bars = axis.bar(
    target_distribution_table[
        "label"
    ],
    target_distribution_table[
        "count"
    ]
)


axis.set_title(
    "Target distribution"
)

axis.set_xlabel(
    "Observed outcome"
)

axis.set_ylabel(
    "Number of applicants"
)


for bar, percentage in zip(
    bars,
    target_distribution_table[
        "percentage"
    ]
):
    axis.text(
        bar.get_x()
        + bar.get_width() / 2,

        bar.get_height(),

        f"{percentage:.2f}%",

        ha="center",
        va="bottom"
    )


figure.tight_layout()


TARGET_DISTRIBUTION_PATH = (
    RESULTS_DIR
    / "02_target_distribution.png"
)


figure.savefig(
    TARGET_DISTRIBUTION_PATH,
    dpi=300,
    bbox_inches="tight"
)


plt.show()
plt.close(figure)

In [ ]:
target_distribution_table.to_csv(
    RESULTS_DIR
    / "02_target_distribution.csv",
    index=False
)

In [ ]:
project_model_comparison.to_csv(
    RESULTS_DIR
    / "04_complete_model_comparison.csv",
    index=False
)


project_model_comparison

In [ ]:
final_boosting_and_ensemble_table.to_csv(
    RESULTS_DIR
    / "05_boosting_cv_and_holdout_performance.csv",
    index=False
)


final_boosting_and_ensemble_table

In [ ]:
threshold_performance_comparison.to_csv(
    RESULTS_DIR
    / "06_threshold_performance.csv",
    index=False
)


final_threshold_summary.to_csv(
    RESULTS_DIR
    / "06_threshold_detailed_summary.csv",
    index=False
)


threshold_performance_comparison

In [ ]:
xgb_mean_absolute_shap.to_csv(
    RESULTS_DIR
    / "12_mean_absolute_shap_importance.csv",
    index=False
)


xgb_grouped_shap_importance.to_csv(
    RESULTS_DIR
    / "13_grouped_shap_importance.csv",
    index=False
)

In [ ]:
gain_importance_source = (
    OUTPUT_DIR
    / "xgboost_component_gain_importance.csv"
)


if gain_importance_source.exists():
    shutil.copy2(
        gain_importance_source,
        RESULTS_DIR
        / "11_xgboost_gain_importance.csv"
    )

else:
    print(
        "Gain-importance CSV was not found. "
        "The gain-importance figure will still be copied below."
    )

In [ ]:
final_model_selection_record.to_csv(
    RESULTS_DIR
    / "14_final_model_selection_record.csv",
    index=False
)


final_model_selection_record

In [ ]:
def find_existing_output(
    exact_filenames=None,
    search_patterns=None
):
    exact_filenames = (
        exact_filenames
        or []
    )

    search_patterns = (
        search_patterns
        or []
    )


    # Check expected exact paths first.
    for filename in exact_filenames:
        candidate_path = (
            OUTPUT_DIR
            / filename
        )

        if candidate_path.exists():
            return candidate_path


    # Search all existing V4 output files,
    # excluding the consolidated results folder.
    matching_paths = []


    for pattern in search_patterns:
        for candidate_path in (
            OUTPUT_DIR.rglob(
                pattern
            )
        ):
            if not candidate_path.is_file():
                continue

            if RESULTS_DIR in (
                candidate_path.parents
            ):
                continue

            matching_paths.append(
                candidate_path
            )


    if not matching_paths:
        return None


    # Prefer the most recently modified matching file.
    return max(
        matching_paths,
        key=lambda path:
        path.stat().st_mtime
    )

In [ ]:
def copy_result_artifact(
    destination_filename,
    exact_filenames=None,
    search_patterns=None,
    required=True
):
    source_path = find_existing_output(
        exact_filenames=(
            exact_filenames
        ),
        search_patterns=(
            search_patterns
        )
    )


    destination_path = (
        RESULTS_DIR
        / destination_filename
    )


    if source_path is None:
        message = (
            "Could not find source artifact for "
            f"{destination_filename}"
        )

        if required:
            print(
                "MISSING:",
                message
            )

        else:
            print(
                "Optional artifact not found:",
                destination_filename
            )

        return {
            "deliverable_file": (
                destination_filename
            ),

            "source_file": None,
            "status": "missing"
        }


    shutil.copy2(
        source_path,
        destination_path
    )


    print(
        "Copied:",
        source_path.name,
        "→",
        destination_path.name
    )


    return {
        "deliverable_file": (
            destination_filename
        ),

        "source_file": str(
            source_path.relative_to(
                OUTPUT_DIR
            )
        ),

        "status": "complete"
    }

In [ ]:
deliverable_copy_records = []

In [ ]:
deliverable_copy_records.append(
    copy_result_artifact(
        destination_filename=(
            "03_selected_correlation_heatmap.png"
        ),

        exact_filenames=[
            "selected_feature_correlation_heatmap.png",
            "selected_correlation_heatmap.png",
            "correlation_heatmap.png"
        ],

        search_patterns=[
            "*selected*correlation*heatmap*.png",
            "*correlation*heatmap*.png",
            "*corr*heatmap*.png"
        ]
    )
)

In [ ]:
deliverable_copy_records.append(
    copy_result_artifact(
        destination_filename=(
            "07_final_roc_curve.png"
        ),

        exact_filenames=[
            "final_ensemble_roc_curve.png",
            "ensemble_roc_curve.png",
            "holdout_roc_curve.png"
        ],

        search_patterns=[
            "*ensemble*roc*curve*.png",
            "*holdout*roc*curve*.png",
            "*roc*curve*.png"
        ]
    )
)

In [ ]:
deliverable_copy_records.append(
    copy_result_artifact(
        destination_filename=(
            "08_final_precision_recall_curve.png"
        ),

        exact_filenames=[
            "final_ensemble_precision_recall_curve.png",
            "ensemble_precision_recall_curve.png",
            "holdout_precision_recall_curve.png"
        ],

        search_patterns=[
            "*ensemble*precision*recall*.png",
            "*holdout*precision*recall*.png",
            "*precision*recall*curve*.png",
            "*pr*curve*.png"
        ]
    )
)

In [ ]:
deliverable_copy_records.append(
    copy_result_artifact(
        destination_filename=(
            "09_final_confusion_matrix.png"
        ),

        exact_filenames=[
            "final_holdout_confusion_matrix.png",
            "holdout_confusion_matrix.png",
            "ensemble_confusion_matrix.png"
        ],

        search_patterns=[
            "*final*confusion*matrix*.png",
            "*holdout*confusion*matrix*.png",
            "*confusion*matrix*.png"
        ]
    )
)

In [ ]:
deliverable_copy_records.append(
    copy_result_artifact(
        destination_filename=(
            "10_xgboost_gain_feature_importance.png"
        ),

        exact_filenames=[
            "xgboost_component_gain_importance.png",
            "xgboost_gain_feature_importance.png",
            "xgboost_feature_importance.png"
        ],

        search_patterns=[
            "*xgboost*gain*importance*.png",
            "*xgboost*feature*importance*.png"
        ]
    )
)

In [ ]:
deliverable_copy_records.append(
    copy_result_artifact(
        destination_filename=(
            "11_xgboost_shap_summary.png"
        ),

        exact_filenames=[
            "xgboost_component_shap_summary.png"
        ],

        search_patterns=[
            "*xgboost*shap*summary*.png"
        ]
    )
)

In [ ]:
deliverable_copy_records.append(
    copy_result_artifact(
        destination_filename=(
            "12_xgboost_grouped_shap_importance.png"
        ),

        exact_filenames=[
            "xgboost_component_grouped_shap_importance.png"
        ],

        search_patterns=[
            "*xgboost*grouped*shap*importance*.png"
        ]
    )
)

In [ ]:
FINAL_RESULTS_SUBMISSION_PATH = (
    RESULTS_DIR
    / "15_home_credit_final_kaggle_submission.csv"
)

In [ ]:
shutil.copy2(
    FINAL_SUBMISSION_PATH,
    FINAL_RESULTS_SUBMISSION_PATH
)


assert FINAL_RESULTS_SUBMISSION_PATH.exists()


print(
    "Kaggle submission copied to:"
)

print(
    FINAL_RESULTS_SUBMISSION_PATH.resolve()
)

In [ ]:
submission_metadata.to_csv(
    RESULTS_DIR
    / "15_kaggle_submission_metadata.csv",
    index=False
)

In [ ]:
requested_deliverables = pd.DataFrame([
    {
        "order": 1,
        "requested_deliverable": (
            "Missing-data summary"
        ),
        "filename": (
            "01_missingness_summary.csv"
        )
    },

    {
        "order": 2,
        "requested_deliverable": (
            "Target-distribution chart"
        ),
        "filename": (
            "02_target_distribution.png"
        )
    },

    {
        "order": 3,
        "requested_deliverable": (
            "Selected correlation heatmap"
        ),
        "filename": (
            "03_selected_correlation_heatmap.png"
        )
    },

    {
        "order": 4,
        "requested_deliverable": (
            "Logistic regression and random-forest "
            "baseline AUC comparison"
        ),
        "filename": (
            "04_complete_model_comparison.csv"
        )
    },

    {
        "order": 5,
        "requested_deliverable": (
            "Final cross-validation and "
            "out-of-sample performance"
        ),
        "filename": (
            "05_boosting_cv_and_holdout_performance.csv"
        )
    },

    {
        "order": 6,
        "requested_deliverable": (
            "Final threshold performance"
        ),
        "filename": (
            "06_threshold_performance.csv"
        )
    },

    {
        "order": 7,
        "requested_deliverable": (
            "ROC curve"
        ),
        "filename": (
            "07_final_roc_curve.png"
        )
    },

    {
        "order": 8,
        "requested_deliverable": (
            "Precision-recall curve"
        ),
        "filename": (
            "08_final_precision_recall_curve.png"
        )
    },

    {
        "order": 9,
        "requested_deliverable": (
            "Confusion matrix"
        ),
        "filename": (
            "09_final_confusion_matrix.png"
        )
    },

    {
        "order": 10,
        "requested_deliverable": (
            "Gain-based feature importance"
        ),
        "filename": (
            "10_xgboost_gain_feature_importance.png"
        )
    },

    {
        "order": 11,
        "requested_deliverable": (
            "SHAP summary plot"
        ),
        "filename": (
            "11_xgboost_shap_summary.png"
        )
    },

    {
        "order": 12,
        "requested_deliverable": (
            "Grouped SHAP importance"
        ),
        "filename": (
            "12_xgboost_grouped_shap_importance.png"
        )
    },

    {
        "order": 13,
        "requested_deliverable": (
            "Final model-selection record"
        ),
        "filename": (
            "14_final_model_selection_record.csv"
        )
    },

    {
        "order": 14,
        "requested_deliverable": (
            "Final Kaggle submission"
        ),
        "filename": (
            "15_home_credit_final_kaggle_submission.csv"
        )
    }
])

In [ ]:
requested_deliverables[
    "exists"
] = (
    requested_deliverables[
        "filename"
    ]
    .map(
        lambda filename:
        (
            RESULTS_DIR
            / filename
        ).exists()
    )
)


requested_deliverables[
    "size_kilobytes"
] = (
    requested_deliverables[
        "filename"
    ]
    .map(
        lambda filename:
        (
            RESULTS_DIR
            / filename
        ).stat().st_size
        / 1024

        if (
            RESULTS_DIR
            / filename
        ).exists()

        else np.nan
    )
)

In [ ]:
requested_deliverables.to_csv(
    RESULTS_DIR
    / "00_DELIVERABLES_INDEX.csv",
    index=False
)


requested_deliverables

In [ ]:
missing_deliverables = (
    requested_deliverables
    .loc[
        ~requested_deliverables[
            "exists"
        ],
        "filename"
    ]
    .tolist()
)

In [ ]:
readme_lines = [
    "# Home Credit V4 Final Results",
    "",
    (
        "This folder contains the final requested "
        "project deliverables."
    ),
    "",
    "## Main deliverables",
    ""
]


for row in (
    requested_deliverables
    .sort_values("order")
    .itertuples(index=False)
):
    status = (
        "Complete"
        if row.exists
        else "MISSING"
    )

    readme_lines.append(
        (
            f"{row.order}. **{row.requested_deliverable}**  \n"
            f"   `{row.filename}` — {status}"
        )
    )


readme_lines.extend([
    "",
    "## Final selected model",
    "",
    (
        "Equal-weight ensemble of XGBoost, "
        "LightGBM, and CatBoost."
    ),
    "",
    (
        f"- Development OOF ROC AUC: "
        f"{V3_SELECTION_RESULTS['Final_ensemble_oof_auc']:.6f}"
    ),
    (
        f"- Development OOF average precision: "
        f"{V3_SELECTION_RESULTS['Final_ensemble_oof_ap']:.6f}"
    ),
    (
        f"- Holdout ROC AUC: "
        f"{final_holdout_auc:.6f}"
    ),
    (
        f"- Holdout average precision: "
        f"{final_holdout_ap:.6f}"
    ),
    (
        f"- Locked F2 threshold: "
        f"{LOCKED_CLASSIFICATION_THRESHOLD:.6f}"
    ),
    "",
    "## Kaggle submission",
    "",
    "`15_home_credit_final_kaggle_submission.csv`",
    "",
    (
        "The Kaggle file contains continuous probabilities. "
        "The classification threshold was not applied."
    )
])

In [ ]:
RESULTS_README_PATH = (
    RESULTS_DIR
    / "00_README.md"
)


RESULTS_README_PATH.write_text(
    "\n".join(
        readme_lines
    ),
    encoding="utf-8"
)

In [ ]:
results_folder_inventory = pd.DataFrame([
    {
        "filename": path.name,
        "extension": path.suffix.lower(),
        "size_kilobytes": (
            path.stat().st_size
            / 1024
        )
    }

    for path in sorted(
        RESULTS_DIR.iterdir()
    )

    if path.is_file()
])

In [ ]:
results_folder_inventory.to_csv(
    RESULTS_DIR
    / "00_RESULTS_FOLDER_INVENTORY.csv",
    index=False
)

In [ ]:
if missing_deliverables:
    print(
        "The following requested deliverables "
        "were not located:"
    )

    for filename in missing_deliverables:
        print(
            " -",
            filename
        )

else:
    print(
        "All requested deliverables are present."
    )

In [ ]:
assert not missing_deliverables, (
    "One or more requested deliverables are missing. "
    "Review the printed filenames and adjust the "
    "source candidate names in the copy cells."
)

In [ ]:
# Recreate XGBoost gain importance from the fitted development model.

xgb_gain_score = (
    xgb_final_model
    .get_booster()
    .get_score(
        importance_type="gain"
    )
)


def resolve_xgb_feature_name(
    booster_feature_name
):
    """
    Convert XGBoost names such as f0, f1, ...
    back to the processed feature names.
    """

    if booster_feature_name in (
        xgb_clean_feature_names
    ):
        return booster_feature_name

    if (
        booster_feature_name.startswith("f")
        and booster_feature_name[1:].isdigit()
    ):
        feature_position = int(
            booster_feature_name[1:]
        )

        if feature_position < len(
            xgb_clean_feature_names
        ):
            return (
                xgb_clean_feature_names[
                    feature_position
                ]
            )

    return booster_feature_name

In [ ]:
xgb_gain_importance = pd.DataFrame({
    "feature": (
        np.asarray(
            xgb_clean_feature_names,
            dtype=str
        )
    )
})


resolved_gain_score = {
    resolve_xgb_feature_name(
        feature_name
    ): float(gain_value)

    for feature_name, gain_value
    in xgb_gain_score.items()
}


xgb_gain_importance[
    "raw_gain"
] = (
    xgb_gain_importance[
        "feature"
    ]
    .map(
        resolved_gain_score
    )
    .fillna(0.0)
)

In [ ]:
total_gain = (
    xgb_gain_importance[
        "raw_gain"
    ].sum()
)


assert total_gain > 0


xgb_gain_importance[
    "importance"
] = (
    xgb_gain_importance[
        "raw_gain"
    ]
    / total_gain
)


xgb_gain_importance = (
    xgb_gain_importance
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(drop=True)
)

In [ ]:
xgb_gain_importance.to_csv(
    OUTPUT_DIR
    / "xgboost_component_gain_importance.csv",
    index=False
)


xgb_gain_importance.to_csv(
    RESULTS_DIR
    / "10_xgboost_gain_importance.csv",
    index=False
)


xgb_gain_importance.head(25)

In [ ]:
assert (
    RESULTS_DIR
    / "10_xgboost_gain_importance.csv"
).exists()


assert np.isclose(
    xgb_gain_importance[
        "importance"
    ].sum(),
    1.0
)


print(
    "Gain-importance CSV saved:"
)

print(
    (
        RESULTS_DIR
        / "10_xgboost_gain_importance.csv"
    ).resolve()
)

In [ ]:
print(
    "=" * 70
)

print(
    "FINAL RESULTS FOLDER COMPLETE"
)

print(
    "=" * 70
)

print(
    "Location:"
)

print(
    RESULTS_DIR.resolve()
)

print()

print(
    "Requested deliverables:",
    len(
        requested_deliverables
    )
)

print(
    "Requested deliverables found:",
    int(
        requested_deliverables[
            "exists"
        ].sum()
    )
)

print(
    "Total files in results folder:",
    len(
        results_folder_inventory
    )
)

print()

print(
    "Start with:"
)

print(
    (
        RESULTS_DIR
        / "00_README.md"
    ).resolve()
)

print()

print(
    "Kaggle submission:"
)

print(
    FINAL_RESULTS_SUBMISSION_PATH.resolve()
)

print(
    "=" * 70
)

Final deliverables conclusion

All requested deliverables have been consolidated into the flat
`outputs/v4/results/` folder.

The folder begins with `00_README.md` and `00_DELIVERABLES_INDEX.csv`, which
identify each requested result and its corresponding filename. The final
Kaggle upload file is:

`15_home_credit_final_kaggle_submission.csv`